In [ ]:
import os, glob, random, math, json
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd

from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
CFG = dict(
    data_root="/kaggle/input/datasets/farahmo/longitudinal-mri-data/Longitudinal MR_data",
    modalities=["FLAIR", "T1", "T2"],
    modality_tags={"FLAIR": "flair", "T1": "t1", "T2": "t2"},
    slice_axis=2,
    img_size=128,
    min_brain_frac=0.02,

    train_frac=0.70,
    val_frac=0.15,

    batch_size=16,
    lr=3e-4,
    epochs=30,
    latent_dim=128,

    w_lesion_bce=1.0,
    w_lesion_dice=1.0,
    w_scl=0.10,

    use_lr_schedule=True,
    lr_min_frac=0.01,
    early_stop_metric="dice",
    early_stopping_patience=6,

    scl_dt=0.005,
    scl_K_train=15,
    scl_K_full=600,
    scl_eps=1e-6,
    scl_xi=1.5,

    scl_lambda_grid=[0.20, 0.40, 0.60, 0.80],
    scl_alpha_grid=[0.02, 0.05, 0.10, 0.20],
    scl_search_batches=8,

    augment=True,
    aug_flip_prob=0.5,
    aug_rot90_prob=0.5,
    aug_intensity_jitter=0.05,

    use_pos_weighting=True,
    use_focal_loss=False,
    focal_gamma=2.0,

    allow_missing_modality=False,

    kfold_n_splits=5,

    num_workers=2,        # DataLoader workers; set to 0 if shared-memory errors occur on Kaggle
    use_amp=True,          # mixed-precision (fp16 autocast + GradScaler) on CUDA

    demographics_csv="/kaggle/input/datasets/farahmo/longitudinal-mri-data/Longitudinal MR_data/long-MR-MS_demographics (1).csv",
    stratify_by="ms_type",
    use_time_conditioning=False,
    seed=SEED,
)

CFG


In [ ]:
_REQUIRED_CFG_KEYS = [
    "data_root", "modalities", "modality_tags", "slice_axis", "img_size", "min_brain_frac",
    "train_frac", "val_frac", "batch_size", "lr", "epochs", "latent_dim",
    "w_lesion_bce", "w_lesion_dice", "w_scl",
    "use_lr_schedule", "lr_min_frac", "early_stop_metric", "early_stopping_patience",
    "scl_dt", "scl_K_train", "scl_K_full", "scl_eps", "scl_xi",
    "scl_lambda_grid", "scl_alpha_grid", "scl_search_batches",
    "augment", "aug_flip_prob", "aug_rot90_prob", "aug_intensity_jitter",
    "use_pos_weighting", "use_focal_loss", "focal_gamma",
    "allow_missing_modality", "kfold_n_splits",
    "num_workers", "use_amp",
    "demographics_csv", "stratify_by", "use_time_conditioning",
    "seed",
]


def validate_cfg(cfg, required_keys=_REQUIRED_CFG_KEYS):
    missing = [k for k in required_keys if k not in cfg]
    if missing:
        raise KeyError(
            f"CFG is missing {len(missing)} required key(s): {missing}\n"
            f"If you edited the CFG cell above, restore these keys (see the original "
            f"CFG definition for their default values) before continuing."
        )
    print(f"CFG validated: all {len(required_keys)} required keys present.")


validate_cfg(CFG)

In [ ]:
N_SEEDS = 5
SEEDS = [42, 43, 44, 45, 46]          
RUN_MULTISEED = True                  
RUN_KFOLD = True                       

CFG["kfold_n_splits"] = 5
CFG["early_stop_metric"] = "dice"

assert len(SEEDS) == N_SEEDS
print(f"N_SEEDS={N_SEEDS}, SEEDS={SEEDS}")
print(f"RUN_MULTISEED={RUN_MULTISEED}  RUN_KFOLD={RUN_KFOLD}  "
      f"kfold_n_splits={CFG['kfold_n_splits']}  early_stop_metric={CFG['early_stop_metric']!r}")


In [ ]:
import re

_PATIENT_DIR_RE = re.compile(r"^patient\s*\d+$", re.IGNORECASE)


def list_patients(root):
  
    found = []
    for dirpath, dirnames, filenames in os.walk(root):
        base = os.path.basename(dirpath.rstrip("/\\"))
        if not _PATIENT_DIR_RE.match(base):
            continue
        has_gt = any("gt" in fn.lower() and (fn.lower().endswith(".nii") or fn.lower().endswith(".nii.gz"))
                      for fn in filenames)
        if has_gt:
            found.append(dirpath)
    return sorted(set(found))


def find_one(patient_dir, *substrings):

    candidates = []
    for f in os.listdir(patient_dir):
        full = os.path.join(patient_dir, f)
        if os.path.isdir(full):
            continue
        low = f.lower()
        if not (low.endswith(".nii") or low.endswith(".nii.gz")):
            continue
        if all(s.lower() in low for s in substrings):
            candidates.append(full)
    if not candidates:
        return None
    candidates.sort(key=len)   
    return candidates[0]


def find_patient_files(pdir, modality_tags, allow_missing_modality=False):
   
    brainmask_path = find_one(pdir, "brainmask")
    gt_path = find_one(pdir, "gt")
    if brainmask_path is None:
        raise FileNotFoundError(f"no *brainmask*.nii[.gz] file in {pdir}")
    if gt_path is None:
        raise FileNotFoundError(f"no *gt*.nii[.gz] file in {pdir}")

    study1_paths, study2_paths = {}, {}
    for m, tag in modality_tags.items():
        p1 = find_one(pdir, "study1", tag)
        p2 = find_one(pdir, "study2", tag)
        if p1 is None:
            if not allow_missing_modality:
                raise FileNotFoundError(f"no study1/{m} ('{tag}') file in {pdir}")
            print(f"  [warn] {os.path.basename(pdir)}: missing study1/{m} -> will zero-fill this channel")
        if p2 is None:
            if not allow_missing_modality:
                raise FileNotFoundError(f"no study2/{m} ('{tag}') file in {pdir}")
            print(f"  [warn] {os.path.basename(pdir)}: missing study2/{m} -> will zero-fill this channel")
        study1_paths[m] = p1
        study2_paths[m] = p2

    return brainmask_path, gt_path, study1_paths, study2_paths


def load_nifti(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return nib.load(path).get_fdata(dtype=np.float32)


def diagnose_data_root(root, modality_tags, max_show=3):

    print(f"Searching recursively under: {root}")
    if not os.path.isdir(root):
        print(f"  !! this path does not exist or is not a directory.")
        return
    top_level = sorted(os.listdir(root))
    print(f"  top-level entries ({len(top_level)}): {top_level[:10]}{' ...' if len(top_level) > 10 else ''}")

    pdirs = list_patients(root)
    print(f"  patient folders found: {len(pdirs)}")
    for p in pdirs[:max_show]:
        print(f"    {p}")
    if len(pdirs) > max_show:
        print(f"    ... and {len(pdirs) - max_show} more")

    if not pdirs:
        print("  No patient folders matched. Checklist:")
        print("   - Is CFG['data_root'] pointing at the right level (above the patientN folders)?")
        print("   - Do patient folder names match /^patient\\\\d+$/ (e.g. 'patient1', not 'Patient_01')?")
        print("   - Does each patient folder contain a '*gt*.nii[.gz]' file directly inside it?")
        return

    print("\n  Per-file check on the first patient folder found:")
    pdir = pdirs[0]
    try:
        bm, gt, s1, s2 = find_patient_files(pdir, modality_tags)
        print(f"    brainmask: {os.path.basename(bm)}")
        print(f"    gt       : {os.path.basename(gt)}")
        for m in modality_tags:
            print(f"    study1/{m}: {os.path.basename(s1[m])}")
            print(f"    study2/{m}: {os.path.basename(s2[m])}")
        print("  -> all required files located successfully for this patient.")
    except FileNotFoundError as e:
        print(f"    !! {e}")
        print(f"    all files actually present in {pdir}:")
        for f in sorted(os.listdir(pdir)):
            print(f"      {f}")


def normalize_volume(vol, mask=None, eps=1e-6):
    '''Robust (1st-99th percentile) intensity normalisation within the brain mask.'''
    vals = vol[mask > 0] if mask is not None else vol.flatten()
    if vals.size == 0:
        return np.zeros_like(vol)
    p1, p99 = np.percentile(vals, [1, 99])
    vol = np.clip(vol, p1, p99)
    vol = (vol - p1) / (p99 - p1 + eps)
    if mask is not None:
        vol = vol * (mask > 0)
    return vol.astype(np.float32)


def resize2d(arr, size):
    t = torch.from_numpy(arr)[None, None].float()
    t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
    return t[0, 0].numpy()


def resize3d(arr, size):
    '''arr: (D, H, W) numpy -> resized to (size, size, size) via trilinear interpolation.'''
    t = torch.from_numpy(arr)[None, None].float()
    t = F.interpolate(t, size=(size, size, size), mode="trilinear", align_corners=False)
    return t[0, 0].numpy()


In [ ]:
def augment_2d(x1, x2, lesion, brain, cfg):
 
    if torch.rand(()).item() < cfg["aug_flip_prob"]:
        x1 = torch.flip(x1, dims=[-1]); x2 = torch.flip(x2, dims=[-1])
        lesion = torch.flip(lesion, dims=[-1]); brain = torch.flip(brain, dims=[-1])
    if torch.rand(()).item() < cfg["aug_flip_prob"]:
        x1 = torch.flip(x1, dims=[-2]); x2 = torch.flip(x2, dims=[-2])
        lesion = torch.flip(lesion, dims=[-2]); brain = torch.flip(brain, dims=[-2])
    if torch.rand(()).item() < cfg["aug_rot90_prob"]:
        k = int(torch.randint(1, 4, (1,)).item())
        x1 = torch.rot90(x1, k, dims=[-2, -1]); x2 = torch.rot90(x2, k, dims=[-2, -1])
        lesion = torch.rot90(lesion, k, dims=[-2, -1]); brain = torch.rot90(brain, k, dims=[-2, -1])

    jitter = cfg["aug_intensity_jitter"]
    if jitter > 0:
        C = x1.shape[0]
        gain1 = 1.0 + (torch.rand(C, 1, 1) * 2 - 1) * jitter
        off1 = (torch.rand(C, 1, 1) * 2 - 1) * jitter
        x1 = (x1 * gain1 + off1).clamp(0, 1)
        gain2 = 1.0 + (torch.rand(C, 1, 1) * 2 - 1) * jitter
        off2 = (torch.rand(C, 1, 1) * 2 - 1) * jitter
        x2 = (x2 * gain2 + off2).clamp(0, 1)

    return x1.contiguous(), x2.contiguous(), lesion.contiguous(), brain.contiguous()


class SliceCache:
    '''Loads NIfTI volumes for the given patient directories and PRECOMPUTES the
    normalized + resized 2D slice arrays ONCE, at construction time -- instead of
    redoing that work (normalize_volume + resize2d, per modality, per slice) on
    every single __getitem__ call, every epoch, the way the original dataset did.

    A SliceCache can be shared by several MSLongitudinalSliceDataset "views" (e.g.
    a k-fold fold's train split and val split, or the same fold retrained under
    several seeds) so a given patient's volumes are only ever read from disk and
    preprocessed once, no matter how many downstream train/val/test partitions of
    that patient pool get built.
    '''

    def __init__(self, patient_dirs, cfg, demographics_df=None):
        self.cfg = cfg
        self.slice_axis = cfg["slice_axis"]
        self.pid_samples = {}   # pid -> list of precomputed per-slice record dicts
        self.samples = []       # master ordered list of (pid, local_idx) across all patients

        for pdir in patient_dirs:
            pid = os.path.basename(pdir)
            try:
                bm_path, gt_path, s1_paths, s2_paths = find_patient_files(
                    pdir, cfg["modality_tags"], allow_missing_modality=cfg["allow_missing_modality"]
                )
                brainmask = load_nifti(bm_path)
                gt = load_nifti(gt_path)

                study1 = {m: (load_nifti(p) if p is not None else np.zeros_like(brainmask))
                          for m, p in s1_paths.items()}
                study2 = {m: (load_nifti(p) if p is not None else np.zeros_like(brainmask))
                          for m, p in s2_paths.items()}
            except FileNotFoundError as e:
                print(f"[skip] {pid}: {e}")
                continue

            dt_days = get_days_between_studies(demographics_df, pid)
            dt_val = np.float32(dt_days / 365.25)

            n_slices = brainmask.shape[self.slice_axis]
            recs = []
            for s in range(n_slices):
                bm_slice = np.take(brainmask, s, axis=self.slice_axis)
                if bm_slice.mean() < cfg["min_brain_frac"]:
                    continue
                gt_slice = np.take(gt, s, axis=self.slice_axis)

                x1_chs, x2_chs = [], []
                for m in cfg["modalities"]:
                    sl1 = normalize_volume(np.take(study1[m], s, axis=self.slice_axis), bm_slice)
                    sl2 = normalize_volume(np.take(study2[m], s, axis=self.slice_axis), bm_slice)
                    x1_chs.append(resize2d(sl1, cfg["img_size"]))
                    x2_chs.append(resize2d(sl2, cfg["img_size"]))

                x1 = np.stack(x1_chs, 0).astype(np.float32)
                x2 = np.stack(x2_chs, 0).astype(np.float32)
                lesion = resize2d((gt_slice > 0).astype(np.float32), cfg["img_size"])[None].astype(np.float32)
                brain = resize2d((bm_slice > 0).astype(np.float32), cfg["img_size"])[None].astype(np.float32)

                recs.append(dict(x1=x1, x2=x2, lesion=lesion, brain=brain, slice_idx=s, dt=dt_val))

            if recs:
                self.pid_samples[pid] = recs
                self.samples.extend((pid, i) for i in range(len(recs)))

        n_patients = len(self.pid_samples)
        print(f"[SliceCache] loaded + precomputed {n_patients} patient(s), "
              f"{len(self.samples)} usable slice(s).")


class MSLongitudinalSliceDataset(Dataset):
    '''Thin, augmentation-only view over a SliceCache. All the expensive work
    (disk I/O, intensity normalization, resizing) already happened once inside
    SliceCache -- __getitem__ here just looks up the precomputed arrays and
    (for train=True) applies random augmentation, which is cheap.

    Two ways to construct it:
      - MSLongitudinalSliceDataset(patient_dirs, cfg, train=..., demographics_df=...)
        with `cache=None` (default): builds its own private SliceCache from the
        given patient directories -- this is a drop-in replacement for the
        original class's call signature, used everywhere outside run_kfold_cv.
      - MSLongitudinalSliceDataset(pids, cfg, train=..., cache=existing_cache)
        reuses an already-built SliceCache and just filters to the given patient
        IDs -- no re-reading or re-preprocessing. This is what run_kfold_cv uses
        so a fold's patients are loaded from disk exactly once, then reused
        across every seed trained on that fold.
    '''

    def __init__(self, patient_dirs, cfg, train=False, demographics_df=None, cache=None):
        self.cfg = cfg
        self.train = train

        if cache is not None:
            self.cache = cache
            wanted = set(patient_dirs)   # here patient_dirs is actually a list of pid strings
        else:
            self.cache = SliceCache(patient_dirs, cfg, demographics_df=demographics_df)
            wanted = set(self.cache.pid_samples.keys())

        self.samples = [(pid, i) for (pid, i) in self.cache.samples if pid in wanted]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pid, i = self.samples[idx]
        rec = self.cache.pid_samples[pid][i]

        x1 = torch.from_numpy(rec["x1"]).clone()
        x2 = torch.from_numpy(rec["x2"]).clone()
        lesion = torch.from_numpy(rec["lesion"]).clone()
        brain = torch.from_numpy(rec["brain"]).clone()

        if self.train and self.cfg.get("augment", False):
            x1, x2, lesion, brain = augment_2d(x1, x2, lesion, brain, self.cfg)

        dt = torch.tensor(float(rec["dt"]), dtype=torch.float32)

        return dict(x1=x1, x2=x2, lesion=lesion, brainmask=brain, pid=pid,
                    slice_idx=rec["slice_idx"], dt=dt)

In [ ]:
def get_days_between_studies(demographics_df, pid, default=365.25):
   
    if demographics_df is None:
        return default
    matches = [c for c in demographics_df.columns if c.lower() == "days_between_studies"]
    if not matches:
        return default
    val = demographics_df[matches[0]].get(pid, None)
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return default
    return float(val)


def load_demographics(csv_path):

    if csv_path is None:
        print("[demographics] CFG['demographics_csv'] is None -- skipping.")
        return None
    if not os.path.exists(csv_path):
        print(f"[demographics] file not found at {csv_path} -- skipping "
              f"(patient split will fall back to plain random).")
        return None

    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    id_col = None
    for candidate in ["patient_id", "patient_ids", "patientid", "id", "patient"]:
        matches = [c for c in df.columns if c.lower() == candidate]
        if matches:
            id_col = matches[0]
            break
    if id_col is None:
        print(f"[demographics] could not find a patient-id column in {csv_path}; "
              f"columns found: {list(df.columns)} -- skipping.")
        return None


    extracted = df[id_col].astype(str).str.extract(r"(\d+)")[0]
    valid = extracted.notna()
    if not valid.all():
        print(f"[demographics] {(~valid).sum()} row(s) had no numeric patient id in "
              f"column '{id_col}' and will be dropped.")
    df = df.loc[valid].copy()
    df["pid"] = extracted.loc[valid].astype(int).apply(lambda x: f"patient{x}")
    df = df.set_index("pid")
    return df


demographics_df = load_demographics(CFG["demographics_csv"])

if demographics_df is not None:
    print(f"Loaded demographics for {len(demographics_df)} patients. Columns: {list(demographics_df.columns)}\n")
    for col in ["sex", "ms_type"]:
        matches = [c for c in demographics_df.columns if c.lower() == col]
        if matches:
            print(f"{col} distribution:")
            print(demographics_df[matches[0]].value_counts())
            print()
    for col in ["days_between_studies", "patient_age_at_first_study"]:
        matches = [c for c in demographics_df.columns if c.lower() == col]
        if matches:
            print(f"{col}: mean={demographics_df[matches[0]].mean():.1f}, "
                  f"std={demographics_df[matches[0]].std():.1f}, "
                  f"range=[{demographics_df[matches[0]].min():.0f}, {demographics_df[matches[0]].max():.0f}]")


In [ ]:
patient_dirs = list_patients(CFG["data_root"])
print(f"Found {len(patient_dirs)} patients under {CFG['data_root']}")

assert len(patient_dirs) > 0, (
    "No patient folders found. Run the diagnose_data_root(...) cell above and fix "
    "CFG['data_root'] / folder naming before continuing."
)


def stratified_patient_split(patient_dirs, cfg, demographics_df):
    
    pids = [os.path.basename(p) for p in patient_dirs]
    strat_col = cfg.get("stratify_by")

    can_stratify = (
        demographics_df is not None
        and strat_col is not None
        and any(c.lower() == strat_col.lower() for c in demographics_df.columns)
    )

    if can_stratify:
        actual_col = [c for c in demographics_df.columns if c.lower() == strat_col.lower()][0]
        labels = pd.Series(pids).map(lambda pid: demographics_df[actual_col].get(pid, None))
        label_counts = labels.value_counts(dropna=False)
        min_class_size = label_counts.min() if len(label_counts) > 0 else 0
        has_missing = labels.isna().any()

        if has_missing or len(label_counts) < 2 or min_class_size < 2:
            print(f"[split] stratification by '{actual_col}' not viable "
                  f"(missing labels: {has_missing}, class sizes: {dict(label_counts)}) "
                  f"-- falling back to plain random split.")
            can_stratify = False

    if can_stratify:
        from sklearn.model_selection import train_test_split
        actual_col = [c for c in demographics_df.columns if c.lower() == strat_col.lower()][0]
        labels = [demographics_df[actual_col].get(pid) for pid in pids]

        test_frac = 1.0 - cfg["train_frac"] - cfg["val_frac"]
        try:
            train_pids, temp_pids, train_labels, temp_labels = train_test_split(
                pids, labels, test_size=(cfg["val_frac"] + test_frac),
                stratify=labels, random_state=cfg["seed"]
            )
            rel_val_frac = cfg["val_frac"] / (cfg["val_frac"] + test_frac)
            val_pids, test_pids, _, _ = train_test_split(
                temp_pids, temp_labels, test_size=(1 - rel_val_frac),
                stratify=temp_labels, random_state=cfg["seed"]
            )
            print(f"[split] stratified by '{actual_col}' (train/val/test class balance preserved).")
        except ValueError as e:
            # A class can have >=2 members overall but still be starved down to 1
            # after the first split (small-N problem) -- fall back rather than crash.
            print(f"[split] stratified split failed even though class counts looked sufficient "
                  f"({e}) -- falling back to plain random split.")
            can_stratify = False

    if not can_stratify:
        rng = random.Random(cfg["seed"])
        shuffled = pids[:]
        rng.shuffle(shuffled)
        n = len(shuffled)
        n_train = max(1, int(cfg["train_frac"] * n))
        n_val = max(1, int(cfg["val_frac"] * n))
        train_pids = shuffled[:n_train]
        val_pids = shuffled[n_train:n_train + n_val]
        test_pids = shuffled[n_train + n_val:]
        print("[split] plain random split (no demographics-based stratification).")

    pid_to_dir = {os.path.basename(p): p for p in patient_dirs}
    return ([pid_to_dir[p] for p in train_pids],
            [pid_to_dir[p] for p in val_pids],
            [pid_to_dir[p] for p in test_pids])


train_p, val_p, test_p = stratified_patient_split(patient_dirs, CFG, demographics_df)

print(f"patients -> train {len(train_p)} / val {len(val_p)} / test {len(test_p)}  "
      f"(patient-level split, no patient appears in more than one of these lists)")

train_ds = MSLongitudinalSliceDataset(train_p, CFG, train=True, demographics_df=demographics_df)
val_ds = MSLongitudinalSliceDataset(val_p, CFG, train=False, demographics_df=demographics_df)
test_ds = MSLongitudinalSliceDataset(test_p, CFG, train=False, demographics_df=demographics_df)
print(f"slices -> train {len(train_ds)} / val {len(val_ds)} / test {len(test_ds)}")

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    assert len(ds) > 0, (
        f"{name} split has 0 usable slices. Either min_brain_frac is too strict, "
        f"or files were skipped -- check the '[skip] ...' messages printed above."
    )

_NW = CFG.get("num_workers", 0)
train_dl = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=_NW,
                       pin_memory=True, persistent_workers=(_NW > 0), drop_last=True,
                       generator=torch.Generator().manual_seed(CFG["seed"]))
val_dl = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=_NW,
                     pin_memory=True, persistent_workers=(_NW > 0))
test_dl = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=_NW,
                      pin_memory=True, persistent_workers=(_NW > 0))

In [ ]:
class SCLFlow(nn.Module):
  

    def __init__(self, lam, alpha, dt=0.006, K=15, eps=1e-6, xi=1.5, clip_u=4.0):
        super().__init__()
        self.lam, self.alpha, self.dt, self.K = lam, alpha, dt, K
        self.eps, self.xi, self.clip_u = eps, xi, clip_u

        lap_kernel = torch.tensor([[0., 1., 0.],
                                    [1., -4., 1.],
                                    [0., 1., 0.]]).view(1, 1, 3, 3)
        self.register_buffer("lap_kernel", lap_kernel)

    def _laplacian(self, x):
        B, C, H, W = x.shape
        x = F.pad(x, (1, 1, 1, 1), mode="replicate")
        k = self.lap_kernel.repeat(C, 1, 1, 1)
        return F.conv2d(x, k, groups=C)

    def _init_u(self, img):
        gx = F.pad(img[:, :, :, 1:] - img[:, :, :, :-1], (0, 1, 0, 0))
        gy = F.pad(img[:, :, 1:, :] - img[:, :, :-1, :], (0, 0, 0, 1))
        grad_mag2 = gx ** 2 + gy ** 2
        denom = grad_mag2.amax(dim=(2, 3), keepdim=True) + self.eps
        return self.xi * grad_mag2 / denom

    def forward(self, x, K=None, return_trajectory=False):
        K = self.K if K is None else K
        u = self._init_u(x)
        traj = [] if return_trajectory else None

        for _ in range(K):
            u = torch.clamp(u, -self.clip_u, self.clip_u)
            R = -2.0 * torch.exp(-2.0 * u) * self._laplacian(u)
            lap_R = self._laplacian(R)
            du = 0.5 * self.dt * ((self.lam - 1.0) * R + self.alpha * lap_R)
            u = u + du
            if return_trajectory:
                traj.append(u.detach())

        u = torch.clamp(u, -self.clip_u, self.clip_u)
        R = -2.0 * torch.exp(-2.0 * u) * self._laplacian(u)
        return (R, traj) if return_trajectory else R


In [ ]:
with torch.no_grad():
    scl_test = SCLFlow(lam=0.60, alpha=0.05, dt=CFG["scl_dt"], K=CFG["scl_K_train"],
                        eps=CFG["scl_eps"], xi=CFG["scl_xi"])
    x = torch.rand(2, 3, 64, 64)
    R = scl_test(x)
    print("input :", x.shape, " curvature output:", R.shape, " finite:", torch.isfinite(R).all().item())


In [ ]:
def scl_lesion_discrimination_auc(scl_module, x, lesion_mask, brain_mask):
    '''AUC of |R| at separating lesion-change voxels from non-lesion voxels, within the brain.'''
    with torch.no_grad():
        R = scl_module(x)
    score = R.abs().mean(dim=1, keepdim=True)  
    score_np = score.cpu().numpy().ravel()
    lesion_np = lesion_mask.cpu().numpy().ravel()
    brain_np = brain_mask.cpu().numpy().ravel()

    valid = brain_np > 0.5

    y = (lesion_np[valid] > 0.5).astype(int)
    s = score_np[valid]
    if y.sum() == 0 or y.sum() == y.size:
        return np.nan
    return roc_auc_score(y, s)


def grid_search_scl_hyperparams(dl, lambda_grid, alpha_grid, cfg, n_batches=8):
    rows = []
    for lam in lambda_grid:
        for alpha in alpha_grid:
            probe = SCLFlow(lam=lam, alpha=alpha, dt=cfg["scl_dt"], K=cfg["scl_K_train"],
                             eps=cfg["scl_eps"], xi=cfg["scl_xi"]).to(device)
            aucs = []
            for i, batch in enumerate(dl):
                if i >= n_batches:
                    break
                x1 = batch["x1"].to(device)
                lesion = batch["lesion"].to(device)
                brain = batch["brainmask"].to(device)
                auc = scl_lesion_discrimination_auc(probe, x1, lesion, brain)
                if not np.isnan(auc):
                    aucs.append(auc)
            mean_auc = float(np.mean(aucs)) if aucs else float("nan")
            rows.append(dict(lam=lam, alpha=alpha, auc=mean_auc, n_valid_batches=len(aucs)))
            print(f"  lambda={lam:.2f}  alpha={alpha:.2f}  ->  lesion-discrimination AUC = {mean_auc:.4f}"
                  f"  (n={len(aucs)}/{n_batches} usable batches)")
    return pd.DataFrame(rows)


print("Running SCL (lambda, alpha) grid search on training data...")
search_results = grid_search_scl_hyperparams(
    train_dl, CFG["scl_lambda_grid"], CFG["scl_alpha_grid"], CFG,
    n_batches=CFG["scl_search_batches"],
)

best_row = search_results.sort_values("auc", ascending=False).iloc[0]
SCL_LAMBDA_STAR = float(best_row["lam"])
SCL_ALPHA_STAR = float(best_row["alpha"])
print(f"\nSelected on this dataset: lambda* = {SCL_LAMBDA_STAR}, alpha* = {SCL_ALPHA_STAR}"
      f"  (AUC = {best_row['auc']:.4f})")

scl_cfg = dict(lam=SCL_LAMBDA_STAR, alpha=SCL_ALPHA_STAR, dt=CFG["scl_dt"],
               K=CFG["scl_K_train"], eps=CFG["scl_eps"], xi=CFG["scl_xi"])
scl_cfg


In [ ]:
pivot = search_results.pivot(index="lam", columns="alpha", values="auc")
plt.figure(figsize=(5.5, 4.5))
plt.imshow(pivot.values, aspect="auto", cmap="viridis")
plt.xticks(range(len(pivot.columns)), [f"{a:.2f}" for a in pivot.columns])
plt.yticks(range(len(pivot.index)), [f"{l:.2f}" for l in pivot.index])
plt.xlabel("alpha"); plt.ylabel("lambda")
plt.title("SCL lesion-discrimination AUC grid (this dataset)")
plt.colorbar(label="AUC")
plt.tight_layout()
plt.show()


In [ ]:
def conv_block(cin, cout, k=3, s=1, p=1):
    return nn.Sequential(
        nn.Conv2d(cin, cout, k, s, p),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )


class Encoder(nn.Module):
    def __init__(self, in_ch, base=32, latent_ch=128):
        super().__init__()
        self.enc1 = nn.Sequential(conv_block(in_ch, base), conv_block(base, base))
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(conv_block(base, base * 2), conv_block(base * 2, base * 2))
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = nn.Sequential(conv_block(base * 2, base * 4), conv_block(base * 4, base * 4))
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = conv_block(base * 4, latent_ch)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        z = self.bottleneck(self.pool3(e3))
        return z, (e1, e2, e3)


class Decoder(nn.Module):
    def __init__(self, latent_ch=128, base=32, out_ch=3, use_skips=True):
        super().__init__()
        self.use_skips = use_skips
        mult = 2 if use_skips else 1
        self.up3 = nn.ConvTranspose2d(latent_ch, base * 4, 2, 2)
        self.dec3 = nn.Sequential(conv_block(base * 4 * mult, base * 4), conv_block(base * 4, base * 4))
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = nn.Sequential(conv_block(base * 2 * mult, base * 2), conv_block(base * 2, base * 2))
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = nn.Sequential(conv_block(base * mult, base), conv_block(base, base))
        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, z, skips=None):
        d3 = self.up3(z)
        if self.use_skips and skips is not None:
            d3 = torch.cat([d3, skips[2]], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        if self.use_skips and skips is not None:
            d2 = torch.cat([d2, skips[1]], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        if self.use_skips and skips is not None:
            d1 = torch.cat([d1, skips[0]], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)


class LatentDynamics(nn.Module):
    '''Predicts z_{t+1} from z_t. Residual update, applied autoregressively so it
    generalises beyond a single hop.

    NOTE ON THIS DATASET: each patient here has exactly two timepoints
    (study1 -> study2), so `rollout(z, n_steps=1)` (equivalently `forward(z)`) is
    all that can be *trained* against. `rollout` with `n_steps > 1` is provided so
    the same module is ready to use if/when a dataset with 3+ longitudinal visits
    per patient becomes available -- it is architecturally supported now, not
    validated against multi-visit data (there isn't any here).'''

    def __init__(self, latent_ch=128):
        super().__init__()
        self.net = nn.Sequential(
            conv_block(latent_ch, latent_ch),
            conv_block(latent_ch, latent_ch),
            nn.Conv2d(latent_ch, latent_ch, 3, 1, 1),
        )

    def forward(self, z):
        return z + self.net(z)

    def rollout(self, z, n_steps=1):
        '''Applies the residual update autoregressively `n_steps` times.
        Returns the FINAL latent state and the list of all intermediate states
        (including the input), so a caller can supervise every hop if multi-visit
        targets are ever available.'''
        states = [z]
        for _ in range(n_steps):
            z = self.forward(z)
            states.append(z)
        return z, states


In [ ]:
class WorldModelBase(nn.Module):
    '''Vanilla world model -- no geometric prior anywhere.'''

    def __init__(self, in_ch, out_ch, latent_ch=128, base=32):
        super().__init__()
        self.encoder = Encoder(in_ch, base=base, latent_ch=latent_ch)
        self.dynamics = LatentDynamics(latent_ch)
        self.decoder = Decoder(latent_ch, base=base, out_ch=out_ch, use_skips=True)
        self.lesion_head = nn.Sequential(
            nn.Conv2d(latent_ch, 32, 3, 1, 1), nn.ReLU(inplace=True), nn.Conv2d(32, 1, 1)
        )

    def forward(self, x1):
        z1, skips = self.encoder(x1)
        z2_hat = self.dynamics(z1)
        x2_hat = self.decoder(z2_hat, skips)
        lesion_logits = F.interpolate(self.lesion_head(z2_hat), size=x1.shape[-2:],
                                       mode="bilinear", align_corners=False)
        return dict(x2_hat=x2_hat, lesion_logits=lesion_logits, z1=z1, z2_hat=z2_hat)


class WorldModelSCLLatent(nn.Module):
    '''Primary SCL variant: SCL flow applied to the encoder's latent feature map,
    fused back in as a geometric regulariser before temporal dynamics.'''

    def __init__(self, in_ch, out_ch, latent_ch=128, base=32, scl_cfg=None):
        super().__init__()
        self.encoder = Encoder(in_ch, base=base, latent_ch=latent_ch)
        self.scl = SCLFlow(**scl_cfg)
        self.latent_proj = nn.Conv2d(latent_ch * 2, latent_ch, 1)
        self.dynamics = LatentDynamics(latent_ch)
        self.decoder = Decoder(latent_ch, base=base, out_ch=out_ch, use_skips=True)
        self.lesion_head = nn.Sequential(
            nn.Conv2d(latent_ch, 32, 3, 1, 1), nn.ReLU(inplace=True), nn.Conv2d(32, 1, 1)
        )

    def forward(self, x1):
        z1_raw, skips = self.encoder(x1)
        R_z1 = self.scl(z1_raw)                                  # curvature of the latent manifold
        z1 = self.latent_proj(torch.cat([z1_raw, R_z1], dim=1))
        z2_hat = self.dynamics(z1)
        x2_hat = self.decoder(z2_hat, skips)
        lesion_logits = F.interpolate(self.lesion_head(z2_hat), size=x1.shape[-2:],
                                       mode="bilinear", align_corners=False)
        return dict(x2_hat=x2_hat, lesion_logits=lesion_logits, z1=z1, z2_hat=z2_hat, R_z1=R_z1)


In [ ]:
class WorldModelSCLInput(nn.Module):
    '''Ablation: SCL applied to the raw input image before encoding.'''

    def __init__(self, in_ch, out_ch, latent_ch=128, base=32, scl_cfg=None):
        super().__init__()
        self.scl = SCLFlow(**scl_cfg)
        self.encoder = Encoder(in_ch * 2, base=base, latent_ch=latent_ch)
        self.dynamics = LatentDynamics(latent_ch)
        self.decoder = Decoder(latent_ch, base=base, out_ch=out_ch, use_skips=True)
        self.lesion_head = nn.Sequential(
            nn.Conv2d(latent_ch, 32, 3, 1, 1), nn.ReLU(inplace=True), nn.Conv2d(32, 1, 1)
        )

    def forward(self, x1):
        R1 = self.scl(x1)
        z1, skips = self.encoder(torch.cat([x1, R1], dim=1))
        z2_hat = self.dynamics(z1)
        x2_hat = self.decoder(z2_hat, skips)
        lesion_logits = F.interpolate(self.lesion_head(z2_hat), size=x1.shape[-2:],
                                       mode="bilinear", align_corners=False)
        return dict(x2_hat=x2_hat, lesion_logits=lesion_logits, z1=z1, z2_hat=z2_hat, R1=R1)


class WorldModelCurvatureDynamics(nn.Module):
    '''Ablation ("Option C"): predicts future curvature explicitly (R1 -> R2_hat)
    in a parallel branch alongside the image latent, then decodes from the image branch.'''

    def __init__(self, in_ch, out_ch, latent_ch=128, base=32, scl_cfg=None):
        super().__init__()
        self.scl = SCLFlow(**scl_cfg)
        self.encoder = Encoder(in_ch, base=base, latent_ch=latent_ch)
        self.curv_encoder = Encoder(in_ch, base=base, latent_ch=latent_ch)
        self.dynamics = LatentDynamics(latent_ch * 2)
        self.split = latent_ch
        self.decoder = Decoder(latent_ch, base=base, out_ch=out_ch, use_skips=True)
        self.lesion_head = nn.Sequential(
            nn.Conv2d(latent_ch, 32, 3, 1, 1), nn.ReLU(inplace=True), nn.Conv2d(32, 1, 1)
        )

    def forward(self, x1):
        R1 = self.scl(x1)
        z1_img, skips = self.encoder(x1)
        z1_curv, _ = self.curv_encoder(R1)
        z1_joint = torch.cat([z1_img, z1_curv], dim=1)
        z2_joint = self.dynamics(z1_joint)
        z2_img_hat, z2_curv_hat = z2_joint[:, :self.split], z2_joint[:, self.split:]
        x2_hat = self.decoder(z2_img_hat, skips)
        lesion_logits = F.interpolate(self.lesion_head(z2_img_hat), size=x1.shape[-2:],
                                       mode="bilinear", align_corners=False)
        return dict(x2_hat=x2_hat, lesion_logits=lesion_logits, z1=z1_joint, z2_hat=z2_joint,
                    R1=R1, R2_hat_latent=z2_curv_hat)


class SCLFlowLearnable(nn.Module):
    '''Same SCL update as `SCLFlow`, but (lambda, alpha) are learned parameters
    rather than fixed constants. Reparameterised through softplus so they remain
    strictly positive regardless of the raw parameter value; initialised at
    (lam_init, alpha_init) -- normally the Section 4b grid-search result -- via the
    softplus inverse, so training starts exactly at that point.'''

    def __init__(self, lam_init, alpha_init, dt=0.006, K=15, eps=1e-6, xi=1.5, clip_u=4.0):
        super().__init__()
        self.dt, self.K = dt, K
        self.eps, self.xi, self.clip_u = eps, xi, clip_u

        def inv_softplus(y):
            y = max(y, 1e-4)
            return math.log(math.expm1(y))

        self.raw_lam = nn.Parameter(torch.tensor(inv_softplus(lam_init), dtype=torch.float32))
        self.raw_alpha = nn.Parameter(torch.tensor(inv_softplus(alpha_init), dtype=torch.float32))

        lap_kernel = torch.tensor([[0., 1., 0.],
                                    [1., -4., 1.],
                                    [0., 1., 0.]]).view(1, 1, 3, 3)
        self.register_buffer("lap_kernel", lap_kernel)

    @property
    def lam(self):
        return F.softplus(self.raw_lam)

    @property
    def alpha(self):
        return F.softplus(self.raw_alpha)

    def _laplacian(self, x):
        B, C, H, W = x.shape
        x = F.pad(x, (1, 1, 1, 1), mode="replicate")
        k = self.lap_kernel.repeat(C, 1, 1, 1)
        return F.conv2d(x, k, groups=C)

    def _init_u(self, img):
        gx = F.pad(img[:, :, :, 1:] - img[:, :, :, :-1], (0, 1, 0, 0))
        gy = F.pad(img[:, :, 1:, :] - img[:, :, :-1, :], (0, 0, 0, 1))
        grad_mag2 = gx ** 2 + gy ** 2
        denom = grad_mag2.amax(dim=(2, 3), keepdim=True) + self.eps
        return self.xi * grad_mag2 / denom

    def forward(self, x, K=None):
        K = self.K if K is None else K
        u = self._init_u(x)
        lam, alpha = self.lam, self.alpha
        for _ in range(K):
            u = torch.clamp(u, -self.clip_u, self.clip_u)
            R = -2.0 * torch.exp(-2.0 * u) * self._laplacian(u)
            lap_R = self._laplacian(R)
            du = 0.5 * self.dt * ((lam - 1.0) * R + alpha * lap_R)
            u = u + du
        u = torch.clamp(u, -self.clip_u, self.clip_u)
        R = -2.0 * torch.exp(-2.0 * u) * self._laplacian(u)
        return R


class WorldModelSCLLatentLearnable(nn.Module):
    '''Same architecture as WorldModelSCLLatent, but using SCLFlowLearnable so
    (lambda, alpha) are refined end-to-end by gradient descent, starting from the
    Section 4b grid-search values rather than being frozen there.'''

    def __init__(self, in_ch, out_ch, latent_ch=128, base=32, lam_init=0.6, alpha_init=0.05, scl_kwargs=None):
        super().__init__()
        scl_kwargs = scl_kwargs or {}
        self.encoder = Encoder(in_ch, base=base, latent_ch=latent_ch)
        self.scl = SCLFlowLearnable(lam_init=lam_init, alpha_init=alpha_init, **scl_kwargs)
        self.latent_proj = nn.Conv2d(latent_ch * 2, latent_ch, 1)
        self.dynamics = LatentDynamics(latent_ch)
        self.decoder = Decoder(latent_ch, base=base, out_ch=out_ch, use_skips=True)
        self.lesion_head = nn.Sequential(
            nn.Conv2d(latent_ch, 32, 3, 1, 1), nn.ReLU(inplace=True), nn.Conv2d(32, 1, 1)
        )

    def forward(self, x1):
        z1_raw, skips = self.encoder(x1)
        R_z1 = self.scl(z1_raw)
        z1 = self.latent_proj(torch.cat([z1_raw, R_z1], dim=1))
        z2_hat = self.dynamics(z1)
        x2_hat = self.decoder(z2_hat, skips)
        lesion_logits = F.interpolate(self.lesion_head(z2_hat), size=x1.shape[-2:],
                                       mode="bilinear", align_corners=False)
        return dict(x2_hat=x2_hat, lesion_logits=lesion_logits, z1=z1, z2_hat=z2_hat, R_z1=R_z1)


In [ ]:
class LatentDynamicsConditioned(nn.Module):
    '''Like LatentDynamics, but FiLM-conditions the residual update on a scalar
    time gap `dt` (e.g. days_between_studies / 365.25). gamma/beta modulate the
    residual delta channel-wise: delta = delta * (1 + gamma(dt)) + beta(dt).'''

    def __init__(self, latent_ch=128):
        super().__init__()
        self.net = nn.Sequential(
            conv_block(latent_ch, latent_ch),
            conv_block(latent_ch, latent_ch),
            nn.Conv2d(latent_ch, latent_ch, 3, 1, 1),
        )
        self.film = nn.Sequential(
            nn.Linear(1, latent_ch), nn.ReLU(inplace=True), nn.Linear(latent_ch, latent_ch * 2)
        )

    def forward(self, z, dt):
        '''dt: [B] or [B, 1] tensor of normalised time gaps (years).'''
        if dt.dim() == 1:
            dt = dt.unsqueeze(-1)
        gamma_beta = self.film(dt)
        gamma, beta = gamma_beta.chunk(2, dim=-1)
        gamma = gamma[:, :, None, None]
        beta = beta[:, :, None, None]
        delta = self.net(z)
        delta = delta * (1 + gamma) + beta
        return z + delta


class WorldModelSCLLatentTimeAware(nn.Module):
    '''Same as WorldModelSCLLatent, but the dynamics step is time-gap conditioned
    via LatentDynamicsConditioned. forward() requires `dt` (from the dataset).'''

    def __init__(self, in_ch, out_ch, latent_ch=128, base=32, scl_cfg=None):
        super().__init__()
        self.encoder = Encoder(in_ch, base=base, latent_ch=latent_ch)
        self.scl = SCLFlow(**scl_cfg)
        self.latent_proj = nn.Conv2d(latent_ch * 2, latent_ch, 1)
        self.dynamics = LatentDynamicsConditioned(latent_ch)
        self.decoder = Decoder(latent_ch, base=base, out_ch=out_ch, use_skips=True)
        self.lesion_head = nn.Sequential(
            nn.Conv2d(latent_ch, 32, 3, 1, 1), nn.ReLU(inplace=True), nn.Conv2d(32, 1, 1)
        )

    def forward(self, x1, dt):
        z1_raw, skips = self.encoder(x1)
        R_z1 = self.scl(z1_raw)
        z1 = self.latent_proj(torch.cat([z1_raw, R_z1], dim=1))
        z2_hat = self.dynamics(z1, dt)
        x2_hat = self.decoder(z2_hat, skips)
        lesion_logits = F.interpolate(self.lesion_head(z2_hat), size=x1.shape[-2:],
                                       mode="bilinear", align_corners=False)
        return dict(x2_hat=x2_hat, lesion_logits=lesion_logits, z1=z1, z2_hat=z2_hat, R_z1=R_z1)


In [ ]:
def dice_loss(logits, target, eps=1e-6):
    probs = torch.sigmoid(logits).flatten(1)
    target = target.flatten(1)
    inter = (probs * target).sum(1)
    union = probs.sum(1) + target.sum(1)
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()


def compute_pos_weight(dl, max_batches=20, cap=100.0):
    '''Estimates BCEWithLogitsLoss pos_weight = (#negative voxels)/(#positive voxels)
    from a few batches of a dataloader. Capped to avoid loss instability when
    lesion-change voxels are extremely rare (e.g. < 1% of the volume).'''
    pos, neg = 0.0, 0.0
    for i, batch in enumerate(dl):
        if i >= max_batches:
            break
        y = batch["lesion"]
        pos += y.sum().item()
        neg += (1.0 - y).sum().item()
    if pos == 0:
        print("[warn] compute_pos_weight: no positive (lesion-change) voxels found in the "
              "sampled batches -- falling back to pos_weight=1.0")
        return 1.0
    return float(min(neg / pos, cap))


def focal_loss_with_logits(logits, target, gamma=2.0, alpha=0.25, eps=1e-6):
    '''Binary focal loss (Lin et al. 2017): down-weights easy voxels so the gradient
    focuses on rare / hard-to-classify lesion-change voxels.'''
    probs = torch.sigmoid(logits).clamp(eps, 1 - eps)
    pt = torch.where(target > 0.5, probs, 1 - probs)
    alpha_t = torch.where(target > 0.5, torch.full_like(probs, alpha), torch.full_like(probs, 1 - alpha))
    loss = -alpha_t * (1 - pt).pow(gamma) * torch.log(pt)
    return loss.mean()


def scl_consistency_loss(x2_hat, x2, scl_module):
    '''Penalise mismatch between the curvature of the predicted and true study2 images.'''
    with torch.no_grad():
        R_true = scl_module(x2)
    R_pred = scl_module(x2_hat.clamp(0, 1))
    return F.mse_loss(R_pred, R_true)


def world_model_loss(out, x2, lesion, cfg, scl_module=None, pos_weight=None):
    recon = F.l1_loss(out["x2_hat"], x2)

    if cfg.get("use_focal_loss", False):
        lesion_bce = focal_loss_with_logits(out["lesion_logits"], lesion, gamma=cfg.get("focal_gamma", 2.0))
    elif cfg.get("use_pos_weighting", False) and pos_weight is not None:
        pw = torch.as_tensor(pos_weight, dtype=torch.float32, device=out["lesion_logits"].device)
        lesion_bce = F.binary_cross_entropy_with_logits(out["lesion_logits"], lesion, pos_weight=pw)
    else:
        lesion_bce = F.binary_cross_entropy_with_logits(out["lesion_logits"], lesion)

    lesion_dc = dice_loss(out["lesion_logits"], lesion)

    total = recon + cfg["w_lesion_bce"] * lesion_bce + cfg["w_lesion_dice"] * lesion_dc
    logs = dict(recon=recon.item(), lesion_bce=lesion_bce.item(), lesion_dice=lesion_dc.item())

    if scl_module is not None:
        l_scl = scl_consistency_loss(out["x2_hat"], x2, scl_module)
        total = total + cfg["w_scl"] * l_scl
        logs["scl_consistency"] = l_scl.item()

    logs["total"] = total.item()
    return total, logs


In [ ]:
def run_epoch(model, dl, cfg, optimizer=None, scl_module_for_loss=None, pos_weight=None,
              scaler=None, use_amp=False):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    agg, n = {}, 0
    for batch in dl:
        x1 = batch["x1"].to(device, non_blocking=True)
        x2 = batch["x2"].to(device, non_blocking=True)
        lesion = batch["lesion"].to(device, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                out = model(x1)
                loss, logs = world_model_loss(out, x2, lesion, cfg, scl_module=scl_module_for_loss,
                                               pos_weight=pos_weight)
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                if use_amp:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

        bs = x1.size(0)
        n += bs
        for k, v in logs.items():
            agg[k] = agg.get(k, 0.0) + v * bs

    return {k: v / n for k, v in agg.items()}


def train_model(model, name, train_dl, val_dl, cfg, epochs=None, scl_module_for_loss=None, pos_weight=None, seed=None):
    
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

    epochs = epochs or cfg["epochs"]
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)

    use_amp = bool(cfg.get("use_amp", False)) and device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    scheduler = None
    if cfg.get("use_lr_schedule", False):
        eta_min = cfg["lr"] * cfg.get("lr_min_frac", 0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=eta_min)

    metric_key = cfg.get("early_stop_metric", "dice")
    patience = cfg.get("early_stopping_patience", None)

    def score_of(va):
        # lower-is-better for both choices: val["lesion_dice"] IS (1 - Dice) already.
        return va["lesion_dice"] if metric_key == "dice" else va["total"]

    history = {"train": [], "val": []}
    best_score, best_state, best_epoch = float("inf"), None, 0
    epochs_since_improve = 0

    for ep in range(1, epochs + 1):
        tr = run_epoch(model, train_dl, cfg, optimizer=opt, scl_module_for_loss=scl_module_for_loss,
                        pos_weight=pos_weight, scaler=scaler, use_amp=use_amp)
        va = run_epoch(model, val_dl, cfg, optimizer=None, scl_module_for_loss=scl_module_for_loss,
                        pos_weight=pos_weight, scaler=scaler, use_amp=use_amp)
        history["train"].append(tr)
        history["val"].append(va)

        score = score_of(va)
        improved = score < best_score - 1e-6
        if improved:
            best_score, best_epoch = score, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1

        lr_now = opt.param_groups[0]["lr"]
        marker = " *" if improved else ""
        print(f"[{name}] epoch {ep:02d}/{epochs}  "
              f"train_loss={tr['total']:.4f}  val_loss={va['total']:.4f}  "
              f"val_dice={1 - va['lesion_dice']:.4f}  lr={lr_now:.2e}{marker}")

        if scheduler is not None:
            scheduler.step()

        if patience is not None and epochs_since_improve >= patience:
            print(f"[{name}] early stopping at epoch {ep} "
                  f"(no improvement in val {metric_key} for {patience} epochs; "
                  f"best was epoch {best_epoch})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

In [ ]:
in_ch = out_ch = len(CFG["modalities"])

model_baseline = WorldModelBase(in_ch, out_ch, latent_ch=CFG["latent_dim"]).to(device)
model_scl_latent = WorldModelSCLLatent(in_ch, out_ch, latent_ch=CFG["latent_dim"], scl_cfg=scl_cfg).to(device)

# fixed (non-trained) SCL module, using the SEARCHED (lambda*, alpha*),
# used only to compute the geometric consistency loss and for post-hoc analysis
scl_module_eval = SCLFlow(**scl_cfg).to(device)


def count_params(m):
    return sum(p.numel() for p in m.parameters())


for name, m in [("Baseline", model_baseline), ("SCL-Latent (primary)", model_scl_latent)]:
    print(f"{name:25s}: {count_params(m):,} params")


In [ ]:
if CFG.get("use_pos_weighting", False) and not CFG.get("use_focal_loss", False):
    pos_weight_2d = compute_pos_weight(train_dl)
    print(f"Estimated BCE pos_weight from training data: {pos_weight_2d:.2f}")
else:
    pos_weight_2d = None
    if CFG.get("use_focal_loss", False):
        print("use_focal_loss=True -> using focal loss instead of pos_weight-BCE")

print("\nTraining Baseline World Model (no SCL)...")
model_baseline, hist_baseline = train_model(
    model_baseline, "Baseline", train_dl, val_dl, CFG, scl_module_for_loss=None, pos_weight=pos_weight_2d
)

print("\nTraining SCL-World Model (latent geometric regulariser, tuned lambda*/alpha*)...")
model_scl_latent, hist_scl = train_model(
    model_scl_latent, "SCL-Latent", train_dl, val_dl, CFG,
    scl_module_for_loss=scl_module_eval, pos_weight=pos_weight_2d
)


In [ ]:
def plot_history(hist_a, hist_b, name_a="Baseline", name_b="SCL-World", key="total"):
    plt.figure(figsize=(6, 4))
    plt.plot([h[key] for h in hist_a["val"]], label=f"{name_a} (val)")
    plt.plot([h[key] for h in hist_b["val"]], label=f"{name_b} (val)")
    plt.xlabel("epoch"); plt.ylabel(key); plt.legend(); plt.title(f"Validation {key} over training")
    plt.tight_layout(); plt.show()

plot_history(hist_baseline, hist_scl, key="total")
plot_history(hist_baseline, hist_scl, key="lesion_dice")


In [ ]:
@torch.no_grad()
def evaluate_model(model, dl, name):

    model.eval()
    per_patient = {}  # pid -> dict(psnr=[...], ssim=[...], dice=[...])

    for batch in dl:
        x1 = batch["x1"].to(device)
        x2 = batch["x2"].to(device)
        lesion = batch["lesion"].to(device)
        pids = batch["pid"]
        out = model(x1)

        x2_hat = out["x2_hat"].clamp(0, 1).cpu().numpy()
        x2_np = x2.cpu().numpy()

        probs = torch.sigmoid(out["lesion_logits"])
        pred = (probs > 0.5).float()
        inter = (pred * lesion).sum(dim=tuple(range(1, lesion.dim())))
        union = pred.sum(dim=tuple(range(1, lesion.dim()))) + lesion.sum(dim=tuple(range(1, lesion.dim())))
        dice_per_sample = ((2 * inter + 1e-6) / (union + 1e-6)).cpu().numpy()

        for b, pid in enumerate(pids):
            entry = per_patient.setdefault(pid, dict(psnr=[], ssim=[], dice=[]))
            for c in range(x2_np.shape[1]):
                entry["psnr"].append(psnr_metric(x2_np[b, c], x2_hat[b, c], data_range=1.0))
                entry["ssim"].append(ssim_metric(x2_np[b, c], x2_hat[b, c], data_range=1.0))
            entry["dice"].append(float(dice_per_sample[b]))

    patient_psnr = [float(np.mean(v["psnr"])) for v in per_patient.values()]
    patient_ssim = [float(np.mean(v["ssim"])) for v in per_patient.values()]
    patient_dice = [float(np.mean(v["dice"])) for v in per_patient.values()]

    return dict(
        model=name,
        n_patients=len(per_patient),
        psnr_mean=float(np.mean(patient_psnr)), psnr_std=float(np.std(patient_psnr)),
        ssim_mean=float(np.mean(patient_ssim)), ssim_std=float(np.std(patient_ssim)),
        dice_mean=float(np.mean(patient_dice)), dice_std=float(np.std(patient_dice)),
    )


@torch.no_grad()
def evaluate_model_slice_weighted(model, dl, name):
    '''DESCRIPTIVE ONLY -- naive per-slice mean (every slice weighted equally,
    so patients with more slices count more). Do not use this for hypothesis
    testing or as the headline number; kept only so you can see how much the two
    weightings differ on your data.'''
    model.eval()
    psnrs, ssims, dices = [], [], []

    for batch in dl:
        x1 = batch["x1"].to(device)
        x2 = batch["x2"].to(device)
        lesion = batch["lesion"].to(device)
        out = model(x1)

        x2_hat = out["x2_hat"].clamp(0, 1).cpu().numpy()
        x2_np = x2.cpu().numpy()
        for b in range(x2_np.shape[0]):
            for c in range(x2_np.shape[1]):
                psnrs.append(psnr_metric(x2_np[b, c], x2_hat[b, c], data_range=1.0))
                ssims.append(ssim_metric(x2_np[b, c], x2_hat[b, c], data_range=1.0))

        probs = torch.sigmoid(out["lesion_logits"])
        pred = (probs > 0.5).float()
        inter = (pred * lesion).sum(dim=(1, 2, 3))
        union = pred.sum(dim=(1, 2, 3)) + lesion.sum(dim=(1, 2, 3))
        dice = ((2 * inter + 1e-6) / (union + 1e-6)).cpu().numpy()
        dices.extend(dice.tolist())

    return dict(
        model=name,
        psnr_mean=float(np.mean(psnrs)), psnr_std=float(np.std(psnrs)),
        ssim_mean=float(np.mean(ssims)), ssim_std=float(np.std(ssims)),
        dice_mean=float(np.mean(dices)), dice_std=float(np.std(dices)),
    )


res_baseline = evaluate_model(model_baseline, test_dl, "Baseline (no SCL)")
res_scl = evaluate_model(model_scl_latent, test_dl, "SCL-World (latent)")

res_baseline_slice = evaluate_model_slice_weighted(model_baseline, test_dl, "Baseline (no SCL)")
res_scl_slice = evaluate_model_slice_weighted(model_scl_latent, test_dl, "SCL-World (latent)")

print(f"Patient-weighted (PRIMARY, n={res_baseline['n_patients']} test patients):")
df = pd.DataFrame([res_baseline, res_scl]).set_index("model")
print(df[["n_patients", "psnr_mean", "psnr_std", "ssim_mean", "ssim_std", "dice_mean", "dice_std"]])

print("\nSlice-weighted (descriptive only -- do not use for significance testing):")
df_slice = pd.DataFrame([res_baseline_slice, res_scl_slice]).set_index("model")
print(df_slice[["psnr_mean", "psnr_std", "ssim_mean", "ssim_std", "dice_mean", "dice_std"]])


In [ ]:
RUN_ABLATIONS = True

if RUN_ABLATIONS:
    model_scl_input = WorldModelSCLInput(in_ch, out_ch, latent_ch=CFG["latent_dim"], scl_cfg=scl_cfg).to(device)
    model_curv_dyn = WorldModelCurvatureDynamics(in_ch, out_ch, latent_ch=CFG["latent_dim"], scl_cfg=scl_cfg).to(device)
    model_scl_learn = WorldModelSCLLatentLearnable(in_ch, out_ch, latent_ch=CFG["latent_dim"],
                                                    lam_init=SCL_LAMBDA_STAR, alpha_init=SCL_ALPHA_STAR).to(device)

    print("Training SCL-Input ablation...")
    model_scl_input, hist_scl_input = train_model(
        model_scl_input, "SCL-Input", train_dl, val_dl, CFG, scl_module_for_loss=scl_module_eval, pos_weight=pos_weight_2d
    )

    print("\nTraining CurvatureDynamics ablation...")
    model_curv_dyn, hist_curv_dyn = train_model(
        model_curv_dyn, "CurvatureDynamics", train_dl, val_dl, CFG, scl_module_for_loss=scl_module_eval, pos_weight=pos_weight_2d
    )

    print("\nTraining learnable-SCL variant...")
    model_scl_learn, hist_scl_learn = train_model(
        model_scl_learn, "SCL-Learnable", train_dl, val_dl, CFG, scl_module_for_loss=scl_module_eval, pos_weight=pos_weight_2d
    )
    print(f"\nLearned (lambda, alpha) after training: "
          f"({model_scl_learn.scl.lam.item():.4f}, {model_scl_learn.scl.alpha.item():.4f})  "
          f"[started from ({SCL_LAMBDA_STAR}, {SCL_ALPHA_STAR})]")

    res_scl_input = evaluate_model(model_scl_input, test_dl, "SCL-Input")
    res_curv_dyn = evaluate_model(model_curv_dyn, test_dl, "CurvatureDynamics")
    res_scl_learn = evaluate_model(model_scl_learn, test_dl, "SCL-Learnable")

    ablation_df = pd.DataFrame([res_baseline, res_scl, res_scl_input, res_curv_dyn, res_scl_learn]).set_index("model")
    print(ablation_df[["psnr_mean", "ssim_mean", "dice_mean"]])
else:
    print("RUN_ABLATIONS is False -- skipping. Set to True above to run the full ablation table.")


In [ ]:
@torch.no_grad()
def scl_energy(R):
    '''Simple proxy for the SCL energy functional: mean squared curvature.'''
    return (R ** 2).mean(dim=tuple(range(1, R.dim())))


@torch.no_grad()
def latent_consistency_report(model, dl, scl_module, n_batches=10):
    model.eval()
    e_true_all, e_pred_all = [], []

    for i, batch in enumerate(dl):
        if i >= n_batches:
            break
        x1 = batch["x1"].to(device)
        x2 = batch["x2"].to(device)
        out = model(x1)

        R_true2 = scl_module(x2)
        R_pred2 = scl_module(out["x2_hat"].clamp(0, 1))
        e_true_all.append(scl_energy(R_true2).cpu())
        e_pred_all.append(scl_energy(R_pred2).cpu())

    e_true = torch.cat(e_true_all).numpy()
    e_pred = torch.cat(e_pred_all).numpy()
    corr = np.corrcoef(e_true, e_pred)[0, 1]
    mae = np.mean(np.abs(e_true - e_pred))
    print(f"SCL-energy correlation (predicted vs. true study2): r={corr:.3f}, MAE={mae:.4f}")
    return e_true, e_pred


e_true, e_pred = latent_consistency_report(model_scl_latent, test_dl, scl_module_eval)


In [ ]:
from scipy.ndimage import label as cc_label
from skimage.morphology import skeletonize


def betti0_error(pred_mask, gt_mask):
    '''|#connected components in pred - #connected components in gt|, for one 2D or
    3D binary mask pair (numpy arrays).'''
    _, n_pred = cc_label(pred_mask > 0.5)
    _, n_gt = cc_label(gt_mask > 0.5)
    return abs(n_pred - n_gt), n_pred, n_gt


def cl_dice(pred_mask, gt_mask, eps=1e-6):
    '''Centerline Dice (Shit et al., 2021). Works for 2D or 3D binary masks.
    skeletonize (2D) is used for 2D masks; for 3D masks each axial slice is
    skeletonized independently (a lightweight approximation of true 3D
    skeletonization, sufficient for a comparative metric between two models).'''
    pred = (pred_mask > 0.5)
    gt = (gt_mask > 0.5)

    if pred.ndim == 2:
        skel_pred = skeletonize(pred)
        skel_gt = skeletonize(gt)
    else:  # 3D: skeletonize slice-by-slice along the first axis
        skel_pred = np.stack([skeletonize(pred[i]) for i in range(pred.shape[0])], axis=0)
        skel_gt = np.stack([skeletonize(gt[i]) for i in range(gt.shape[0])], axis=0)

    t_prec = (skel_pred & gt).sum() / (skel_pred.sum() + eps)
    t_sens = (skel_gt & pred).sum() / (skel_gt.sum() + eps)
    return float(2 * t_prec * t_sens / (t_prec + t_sens + eps))


@torch.no_grad()
def evaluate_topology(model, dl, name, max_samples=40):
    '''Runs betti0_error and cl_dice over up to `max_samples` individual lesion-change
    predictions (kept small by default -- skeletonization is not free).'''
    model.eval()
    betti_errors, cldices = [], []
    n_seen = 0

    for batch in dl:
        if n_seen >= max_samples:
            break
        x1 = batch["x1"].to(device)
        lesion = batch["lesion"].cpu().numpy()
        out = model(x1)
        pred = (torch.sigmoid(out["lesion_logits"]) > 0.5).cpu().numpy()

        for b in range(pred.shape[0]):
            if n_seen >= max_samples:
                break
            p = pred[b, 0]
            g = lesion[b, 0]
            if g.sum() == 0 and p.sum() == 0:
                n_seen += 1
                continue  # skeletonize on an all-empty mask is degenerate; skip
            err, _, _ = betti0_error(p, g)
            betti_errors.append(err)
            cldices.append(cl_dice(p, g))
            n_seen += 1

    return dict(
        model=name,
        betti0_error_mean=float(np.mean(betti_errors)) if betti_errors else float("nan"),
        betti0_error_std=float(np.std(betti_errors)) if betti_errors else float("nan"),
        cldice_mean=float(np.mean(cldices)) if cldices else float("nan"),
        cldice_std=float(np.std(cldices)) if cldices else float("nan"),
        n_evaluated=len(betti_errors),
    )


topo_baseline = evaluate_topology(model_baseline, test_dl, "Baseline (no SCL)")
topo_scl = evaluate_topology(model_scl_latent, test_dl, "SCL-World (latent)")

topo_df = pd.DataFrame([topo_baseline, topo_scl]).set_index("model")
topo_df


In [ ]:
from scipy.stats import wilcoxon


@torch.no_grad()
def evaluate_paired_patient_level(model_a, model_b, dl):
    '''VALID for hypothesis testing. Computes per-slice metrics for both models,
    groups by patient, averages within each patient, and returns one paired
    (Baseline, SCL) value PER PATIENT -- the correct unit of analysis, since
    slices from the same patient are correlated, not independent.'''
    model_a.eval(); model_b.eval()
    per_patient = {}  # pid -> dict(psnr_a=[], psnr_b=[], ssim_a=[], ssim_b=[], dice_a=[], dice_b=[])

    for batch in dl:
        x1 = batch["x1"].to(device)
        x2 = batch["x2"].to(device)
        lesion = batch["lesion"].to(device)
        pids = batch["pid"]
        oa = model_a(x1)
        ob = model_b(x1)

        xa = oa["x2_hat"].clamp(0, 1).cpu().numpy()
        xb = ob["x2_hat"].clamp(0, 1).cpu().numpy()
        xt = x2.cpu().numpy()

        pa = (torch.sigmoid(oa["lesion_logits"]) > 0.5).float()
        pb = (torch.sigmoid(ob["lesion_logits"]) > 0.5).float()
        dims = tuple(range(1, lesion.dim()))
        inter_a = (pa * lesion).sum(dim=dims); union_a = pa.sum(dim=dims) + lesion.sum(dim=dims)
        inter_b = (pb * lesion).sum(dim=dims); union_b = pb.sum(dim=dims) + lesion.sum(dim=dims)
        dice_a = ((2 * inter_a + 1e-6) / (union_a + 1e-6)).cpu().numpy()
        dice_b = ((2 * inter_b + 1e-6) / (union_b + 1e-6)).cpu().numpy()

        for b, pid in enumerate(pids):
            entry = per_patient.setdefault(
                pid, dict(psnr_a=[], psnr_b=[], ssim_a=[], ssim_b=[], dice_a=[], dice_b=[])
            )
            for c in range(xt.shape[1]):
                entry["psnr_a"].append(psnr_metric(xt[b, c], xa[b, c], data_range=1.0))
                entry["psnr_b"].append(psnr_metric(xt[b, c], xb[b, c], data_range=1.0))
                entry["ssim_a"].append(ssim_metric(xt[b, c], xa[b, c], data_range=1.0))
                entry["ssim_b"].append(ssim_metric(xt[b, c], xb[b, c], data_range=1.0))
            entry["dice_a"].append(float(dice_a[b]))
            entry["dice_b"].append(float(dice_b[b]))

    out = {k: [] for k in ["psnr_a", "psnr_b", "ssim_a", "ssim_b", "dice_a", "dice_b"]}
    for pid, entry in per_patient.items():
        for k in out:
            out[k].append(float(np.mean(entry[k])))

    return {k: np.array(v) for k, v in out.items()}, list(per_patient.keys())


@torch.no_grad()
def evaluate_paired_slice_level(model_a, model_b, dl):
    '''DESCRIPTIVE ONLY -- per-slice pairing. Do NOT use for significance testing:
    slices from the same patient are correlated, not independent, so p-values and
    CIs computed on this are invalid (pseudoreplication).'''
    model_a.eval(); model_b.eval()
    out = {"psnr_a": [], "psnr_b": [], "ssim_a": [], "ssim_b": [], "dice_a": [], "dice_b": []}

    for batch in dl:
        x1 = batch["x1"].to(device)
        x2 = batch["x2"].to(device)
        lesion = batch["lesion"].to(device)
        oa = model_a(x1)
        ob = model_b(x1)

        xa = oa["x2_hat"].clamp(0, 1).cpu().numpy()
        xb = ob["x2_hat"].clamp(0, 1).cpu().numpy()
        xt = x2.cpu().numpy()
        for b in range(xt.shape[0]):
            for c in range(xt.shape[1]):
                out["psnr_a"].append(psnr_metric(xt[b, c], xa[b, c], data_range=1.0))
                out["psnr_b"].append(psnr_metric(xt[b, c], xb[b, c], data_range=1.0))
                out["ssim_a"].append(ssim_metric(xt[b, c], xa[b, c], data_range=1.0))
                out["ssim_b"].append(ssim_metric(xt[b, c], xb[b, c], data_range=1.0))

        pa = (torch.sigmoid(oa["lesion_logits"]) > 0.5).float()
        pb = (torch.sigmoid(ob["lesion_logits"]) > 0.5).float()
        for pred, key in [(pa, "dice_a"), (pb, "dice_b")]:
            inter = (pred * lesion).sum(dim=(1, 2, 3))
            union = pred.sum(dim=(1, 2, 3)) + lesion.sum(dim=(1, 2, 3))
            dice = ((2 * inter + 1e-6) / (union + 1e-6)).cpu().numpy()
            out[key].extend(dice.tolist())

    return {k: np.array(v) for k, v in out.items()}


def paired_bootstrap_ci(diffs, n_boot=10000, ci=95, seed=0):
    '''Percentile bootstrap CI on the mean of `diffs` (e.g. per-patient SCL - Baseline).'''
    rng = np.random.default_rng(seed)
    n = len(diffs)
    boot_means = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_means[i] = diffs[idx].mean()
    lo, hi = np.percentile(boot_means, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return float(diffs.mean()), float(lo), float(hi)


def holm_bonferroni(p_values):
    '''Holm-Bonferroni step-down correction. Returns adjusted p-values in the same
    order as the input (not sorted), each capped at 1.0 and monotonically
    non-decreasing in the sorted order (standard Holm guarantee).'''
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    adjusted_sorted = np.empty(n)
    running_max = 0.0
    for i, idx in enumerate(order):
        adj = (n - i) * p_values[idx]
        running_max = max(running_max, adj)
        adjusted_sorted[i] = min(running_max, 1.0)
    adjusted = np.empty(n)
    adjusted[order] = adjusted_sorted
    return adjusted


def paired_significance_report(a, b, metric_name, n_patients=None):
    '''a, b: paired arrays for (Baseline, SCL-World) -- one value per PATIENT if
    called on evaluate_paired_patient_level output (as it should be for inference).'''
    diffs = b - a
    mean_diff, lo, hi = paired_bootstrap_ci(diffs)
    if np.allclose(a, b):
        p = 1.0
    else:
        try:
            _, p = wilcoxon(a, b)
        except ValueError:
            p = float("nan")
    n_str = f" (n={n_patients} patients)" if n_patients is not None else f" (n={len(a)})"
    print(f"{metric_name:6s}{n_str}: mean(SCL - Baseline) = {mean_diff:+.4f}   "
          f"95% CI [{lo:+.4f}, {hi:+.4f}]   Wilcoxon p = {p:.4g}")
    return dict(metric=metric_name, n=len(a), mean_diff=mean_diff, ci_lo=lo, ci_hi=hi, wilcoxon_p=p)


paired_patient, patient_ids_tested = evaluate_paired_patient_level(model_baseline, model_scl_latent, test_dl)
n_test_patients = len(patient_ids_tested)
print(f"PATIENT-LEVEL test (valid for inference; n = {n_test_patients} test patients):")
if n_test_patients < 5:
    print(f"  [caution] only {n_test_patients} test patients -- any test here has very "
          f"low power; a non-significant p-value is NOT evidence of no effect at this n.")

stats_rows = [
    paired_significance_report(paired_patient["psnr_a"], paired_patient["psnr_b"], "PSNR", n_test_patients),
    paired_significance_report(paired_patient["ssim_a"], paired_patient["ssim_b"], "SSIM", n_test_patients),
    paired_significance_report(paired_patient["dice_a"], paired_patient["dice_b"], "Dice", n_test_patients),
]
raw_ps = [r["wilcoxon_p"] for r in stats_rows]
adjusted_ps = holm_bonferroni(raw_ps)
for r, p_adj in zip(stats_rows, adjusted_ps):
    r["wilcoxon_p_holm"] = float(p_adj)
    r["significant_after_correction"] = bool(p_adj < 0.05 and (r["ci_lo"] > 0 or r["ci_hi"] < 0))

stats_df = pd.DataFrame(stats_rows).set_index("metric")
print("\nWith Holm-Bonferroni correction across the 3 metrics:")
print(stats_df[["n", "mean_diff", "ci_lo", "ci_hi", "wilcoxon_p", "wilcoxon_p_holm", "significant_after_correction"]])

print("\n--- Descriptive-only slice-level pairing (NOT for significance claims) ---")
paired_slice = evaluate_paired_slice_level(model_baseline, model_scl_latent, test_dl)
print(f"n slice-level pairs: PSNR/SSIM={len(paired_slice['psnr_a'])}, Dice={len(paired_slice['dice_a'])}")
for m in ["psnr", "ssim", "dice"]:
    d = paired_slice[f"{m}_b"] - paired_slice[f"{m}_a"]
    print(f"  {m}: slice-level mean diff = {d.mean():+.4f} (descriptive only; "
          f"do not compute a p-value on this -- see caution above)")


In [ ]:
RUN_TIME_AWARE_DEMO = True

if RUN_TIME_AWARE_DEMO:
    model_time_aware = WorldModelSCLLatentTimeAware(
        in_ch, out_ch, latent_ch=CFG["latent_dim"], scl_cfg=scl_cfg
    ).to(device)
    opt_ta = torch.optim.Adam(model_time_aware.parameters(), lr=CFG["lr"], weight_decay=1e-4)

    print("Training time-aware SCL-World variant (demo)...")
    for ep in range(1, CFG["epochs"] + 1):
        model_time_aware.train()
        total_loss, n = 0.0, 0
        for batch in train_dl:
            x1 = batch["x1"].to(device)
            x2 = batch["x2"].to(device)
            lesion = batch["lesion"].to(device)
            dt = batch["dt"].to(device)

            out = model_time_aware(x1, dt)
            loss, _ = world_model_loss(out, x2, lesion, CFG, scl_module=scl_module_eval, pos_weight=pos_weight_2d)

            opt_ta.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_time_aware.parameters(), 1.0)
            opt_ta.step()

            total_loss += loss.item() * x1.size(0)
            n += x1.size(0)
        print(f"[TimeAware] epoch {ep:02d}/{CFG['epochs']}  train_loss={total_loss / n:.4f}")

    @torch.no_grad()
    def evaluate_time_aware(model, dl, name):
        model.eval()
        psnrs = []
        for batch in dl:
            x1 = batch["x1"].to(device)
            x2 = batch["x2"].to(device)
            dt = batch["dt"].to(device)
            out = model(x1, dt)
            x2_hat = out["x2_hat"].clamp(0, 1).cpu().numpy()
            x2_np = x2.cpu().numpy()
            for b in range(x2_np.shape[0]):
                for c in range(x2_np.shape[1]):
                    psnrs.append(psnr_metric(x2_np[b, c], x2_hat[b, c], data_range=1.0))
        mean_psnr = float(np.mean(psnrs))
        print(f"{name}: PSNR = {mean_psnr:.2f} dB")
        return mean_psnr

    evaluate_time_aware(model_time_aware, test_dl, "SCL-World (time-aware)")
else:
    print("RUN_TIME_AWARE_DEMO is False -- skipping. Set to True above to train the time-gap-conditioned variant.")


In [ ]:
@torch.no_grad()
def evaluate_lesion_volume(model, dl, name):
    model.eval()
    per_patient_pred, per_patient_true = {}, {}

    for batch in dl:
        x1 = batch["x1"].to(device)
        lesion = batch["lesion"].to(device)
        pids = batch["pid"]
        out = model(x1)
        pred = (torch.sigmoid(out["lesion_logits"]) > 0.5).float()

        pred_voxels = pred.sum(dim=tuple(range(1, pred.dim()))).cpu().numpy()
        true_voxels = lesion.sum(dim=tuple(range(1, lesion.dim()))).cpu().numpy()

        for b, pid in enumerate(pids):
            per_patient_pred[pid] = per_patient_pred.get(pid, 0.0) + float(pred_voxels[b])
            per_patient_true[pid] = per_patient_true.get(pid, 0.0) + float(true_voxels[b])

    pids_sorted = sorted(per_patient_pred.keys())
    pred_arr = np.array([per_patient_pred[p] for p in pids_sorted])
    true_arr = np.array([per_patient_true[p] for p in pids_sorted])

    mae = float(np.mean(np.abs(pred_arr - true_arr)))
    if len(pred_arr) >= 2 and np.std(pred_arr) > 0 and np.std(true_arr) > 0:
        corr = float(np.corrcoef(pred_arr, true_arr)[0, 1])
    else:
        corr = float("nan")

    return dict(model=name, n_patients=len(pids_sorted), volume_mae_voxels=mae, volume_corr=corr,
                pred_volumes=pred_arr, true_volumes=true_arr, pids=pids_sorted)


vol_baseline = evaluate_lesion_volume(model_baseline, test_dl, "Baseline (no SCL)")
vol_scl = evaluate_lesion_volume(model_scl_latent, test_dl, "SCL-World (latent)")

vol_df = pd.DataFrame([
    {k: v for k, v in d.items() if k not in ("pred_volumes", "true_volumes", "pids")}
    for d in [vol_baseline, vol_scl]
]).set_index("model")
print(vol_df)

fig, ax = plt.subplots(figsize=(5, 5))
lims = [0, max(vol_baseline["true_volumes"].max(), vol_scl["true_volumes"].max(),
               vol_baseline["pred_volumes"].max(), vol_scl["pred_volumes"].max()) * 1.1 + 1]
ax.plot(lims, lims, "k--", alpha=0.4, label="perfect agreement")
ax.scatter(vol_baseline["true_volumes"], vol_baseline["pred_volumes"], color="#888888", label="Baseline")
ax.scatter(vol_scl["true_volumes"], vol_scl["pred_volumes"], color="#0053a5", label="SCL-World")
ax.set_xlabel("True total lesion-change volume (voxels)")
ax.set_ylabel("Predicted total lesion-change volume (voxels)")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.legend()
ax.set_title("Per-patient lesion volume: predicted vs. true")
plt.tight_layout()
plt.show()

# Paired comparison: for each patient, is SCL-World's volume estimate closer to
# the truth than Baseline's? (paired by patient, valid unit of analysis)
common_pids = [p for p in vol_baseline["pids"] if p in vol_scl["pids"]]
if len(common_pids) >= 2:
    base_err = np.abs(vol_baseline["pred_volumes"] - vol_baseline["true_volumes"])
    scl_err = np.abs(vol_scl["pred_volumes"] - vol_scl["true_volumes"])
    diffs = scl_err - base_err  # negative = SCL closer to truth
    mean_diff, lo, hi = paired_bootstrap_ci(diffs)
    print(f"\nPer-patient |volume error| difference (SCL - Baseline): {mean_diff:+.2f} voxels, "
          f"95% CI [{lo:+.2f}, {hi:+.2f}]")
    print("(negative mean/CI => SCL-World's volume estimates are closer to ground truth)")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = [("psnr_mean", "psnr_std", "PSNR (dB)"),
           ("ssim_mean", "ssim_std", "SSIM"),
           ("dice_mean", "dice_std", "Lesion-change Dice")]

for ax, (mkey, skey, title) in zip(axes, metrics):
    vals = [res_baseline[mkey], res_scl[mkey]]
    errs = [res_baseline[skey], res_scl[skey]]
    ax.bar(["Baseline", "SCL-World"], vals, yerr=errs, capsize=4, color=["#888888", "#0053a5"])
    ax.set_title(title)

fig.suptitle(f"Primary split (single seed) -- patient-level mean ± std "
             f"(n={res_baseline['n_patients']} test patients). "
             f"See Section 12b/13b for the multi-seed / k-fold aggregated versions.",
             fontsize=9, y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
@torch.no_grad()
def show_examples_2d(model_a, model_b, dl, scl_module, n=3, names=("Baseline", "SCL-World")):
    model_a.eval(); model_b.eval()
    batch = next(iter(dl))
    x1 = batch["x1"].to(device)
    x2 = batch["x2"].to(device)
    lesion = batch["lesion"]

    out_a = model_a(x1)
    out_b = model_b(x1)
    R1 = scl_module(x1)

    n = min(n, x1.shape[0])
    fig, axes = plt.subplots(n, 6, figsize=(18, 3 * n))
    if n == 1:
        axes = axes[None, :]

    col_titles = ["Study1 (input)", "Study2 (GT)", f"{names[0]} pred",
                  f"{names[1]} pred", "SCL curvature R(study1)", "Lesion change (GT)"]

    for i in range(n):
        imgs = [x1[i, 0].cpu(), x2[i, 0].cpu(),
                out_a["x2_hat"][i, 0].clamp(0, 1).cpu(),
                out_b["x2_hat"][i, 0].clamp(0, 1).cpu(),
                R1[i, 0].cpu(), lesion[i, 0].cpu()]
        for j, im in enumerate(imgs):
            axes[i, j].imshow(im, cmap="magma" if j == 4 else "gray")
            axes[i, j].axis("off")
            if i == 0:
                axes[i, j].set_title(col_titles[j], fontsize=10)

    plt.tight_layout()
    plt.show()


show_examples_2d(model_baseline, model_scl_latent, test_dl, scl_module_eval)


In [ ]:
multiseed_records = []

def build_seeded_train_dl(seed):
    # Fresh, non-persistent loader per seed: a shared/persistent train_dl
    # would keep one continuous worker RNG stream across every seed and
    # model in the sweep, decoupling the augmentation/shuffle order actually
    # used from the seed label attached to the results. Rebuilding here ties
    # that stream to `seed` and resets it at the start of each seed.
    nw = CFG.get("num_workers", 0)
    return DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=nw,
                       pin_memory=True, persistent_workers=False, drop_last=True,
                       generator=torch.Generator().manual_seed(seed))

if RUN_MULTISEED:
    for s in SEEDS:
        print(f"\n========== seed {s} ==========")
        train_dl_s = build_seeded_train_dl(s)

        # Re-run the SCL hyperparameter search under this seed (Section 4b criterion),
        # rather than reusing the (lambda*, alpha*) fit once on the primary run.
        random.seed(s); np.random.seed(s); torch.manual_seed(s)
        search_results_s = grid_search_scl_hyperparams(
            train_dl_s, CFG["scl_lambda_grid"], CFG["scl_alpha_grid"], CFG,
            n_batches=CFG["scl_search_batches"],
        )
        best_row_s = search_results_s.sort_values("auc", ascending=False).iloc[0]
        scl_cfg_s = dict(lam=float(best_row_s["lam"]), alpha=float(best_row_s["alpha"]),
                          dt=CFG["scl_dt"], K=CFG["scl_K_train"], eps=CFG["scl_eps"], xi=CFG["scl_xi"])
        print(f"  [seed {s}] selected lambda*={scl_cfg_s['lam']}, alpha*={scl_cfg_s['alpha']}")

        m_base_s = WorldModelBase(in_ch, out_ch, latent_ch=CFG["latent_dim"]).to(device)
        m_scl_s = WorldModelSCLLatent(in_ch, out_ch, latent_ch=CFG["latent_dim"], scl_cfg=scl_cfg_s).to(device)
        scl_eval_s = SCLFlow(**scl_cfg_s).to(device)

        pw_s = compute_pos_weight(train_dl_s) if CFG.get("use_pos_weighting", False) else None

        m_base_s, _ = train_model(m_base_s, f"seed{s}-Baseline", train_dl_s, val_dl, CFG,
                                   pos_weight=pw_s, seed=s)
        m_scl_s, _ = train_model(m_scl_s, f"seed{s}-SCL", train_dl_s, val_dl, CFG,
                                  scl_module_for_loss=scl_eval_s, pos_weight=pw_s, seed=s)

        rb_s = evaluate_model(m_base_s, test_dl, "Baseline")
        rs_s = evaluate_model(m_scl_s, test_dl, "SCL-World")
        rb_s["seed"] = s; rs_s["seed"] = s
        multiseed_records.append(rb_s)
        multiseed_records.append(rs_s)

    multiseed_df = pd.DataFrame(multiseed_records)
    print("\nPer-seed patient-weighted results:")
    print(multiseed_df[["model", "seed", "psnr_mean", "ssim_mean", "dice_mean"]])

    multiseed_summary = multiseed_df.groupby("model")[["psnr_mean", "ssim_mean", "dice_mean"]].agg(["mean", "std"])
    print(f"\nAcross-seed summary (n={N_SEEDS} seeds, mean +/- std, patient-level PRIMARY metric):")
    print(multiseed_summary)
else:
    multiseed_df, multiseed_summary = None, None
    print("RUN_MULTISEED is False -- skipping. Set to True in Section 2b for the full multi-seed table.")


## Multi-seed study, extended: add CurvatureDynamics

Adds `WorldModelCurvatureDynamics` to the same multi-seed sweep used for Baseline and SCL-Latent (Section 12b), under the *same* seeds and the *same* per-seed re-selected `(lambda*, alpha*)`. Until this cell (and its k-fold counterpart below) has been run, CurvatureDynamics's Table 6 lead (Dice 0.298 vs 0.265, n=3 test patients, single split) is not validated evidence that it beats SCL-Latent -- it is a single-split number that may or may not survive resampling, exactly like SCL-Latent's own accuracy numbers didn't survive the multi-seed study. This cell finds out, rather than assuming.

In [ ]:
curvature_multiseed_records = []

if RUN_MULTISEED:
    for s in SEEDS:
        print(f"\n========== [CurvatureDynamics] seed {s} ==========")
        train_dl_s = build_seeded_train_dl(s)  # defined in the Baseline/SCL-Latent sweep above

        random.seed(s); np.random.seed(s); torch.manual_seed(s)
        search_results_cs = grid_search_scl_hyperparams(
            train_dl_s, CFG["scl_lambda_grid"], CFG["scl_alpha_grid"], CFG,
            n_batches=CFG["scl_search_batches"],
        )
        best_row_cs = search_results_cs.sort_values("auc", ascending=False).iloc[0]
        scl_cfg_cs = dict(lam=float(best_row_cs["lam"]), alpha=float(best_row_cs["alpha"]),
                           dt=CFG["scl_dt"], K=CFG["scl_K_train"], eps=CFG["scl_eps"], xi=CFG["scl_xi"])
        print(f"  [seed {s}] selected lambda*={scl_cfg_cs['lam']}, alpha*={scl_cfg_cs['alpha']}")

        m_curv_s = WorldModelCurvatureDynamics(in_ch, out_ch, latent_ch=CFG["latent_dim"],
                                                scl_cfg=scl_cfg_cs).to(device)
        scl_eval_cs = SCLFlow(**scl_cfg_cs).to(device)
        pw_cs = compute_pos_weight(train_dl_s) if CFG.get("use_pos_weighting", False) else None

        m_curv_s, _ = train_model(m_curv_s, f"seed{s}-CurvatureDynamics", train_dl_s, val_dl, CFG,
                                   scl_module_for_loss=scl_eval_cs, pos_weight=pw_cs, seed=s)

        rc_s = evaluate_model(m_curv_s, test_dl, "CurvatureDynamics")
        rc_s["seed"] = s
        curvature_multiseed_records.append(rc_s)

    curvature_multiseed_df = pd.DataFrame(curvature_multiseed_records)
    print("\nPer-seed patient-weighted results [CurvatureDynamics]:")
    print(curvature_multiseed_df[["model", "seed", "psnr_mean", "ssim_mean", "dice_mean"]])

    curvature_multiseed_summary = curvature_multiseed_df.groupby("model")[
        ["psnr_mean", "ssim_mean", "dice_mean"]].agg(["mean", "std"])
    print(f"\nAcross-seed summary (n={N_SEEDS} seeds, mean +/- std) [CurvatureDynamics]:")
    print(curvature_multiseed_summary)

    # Combined view: all three models side by side on the SAME multi-seed protocol.
    combined_multiseed_summary = pd.concat(
        [multiseed_summary, curvature_multiseed_summary]
    ) if multiseed_df is not None else curvature_multiseed_summary
    print("\n=== Combined multi-seed summary: Baseline vs SCL-Latent vs CurvatureDynamics ===")
    print(combined_multiseed_summary)
else:
    curvature_multiseed_df, curvature_multiseed_summary, combined_multiseed_summary = None, None, None
    print("RUN_MULTISEED is False -- skipping CurvatureDynamics multi-seed sweep too.")


In [ ]:
def add_gaussian_noise(x, sigma):
    return (x + sigma * torch.randn_like(x)).clamp(0, 1)


def add_salt_pepper(x, density):
    mask = torch.rand_like(x)
    out = x.clone()
    out[mask < density / 2] = 0.0
    out[mask > 1 - density / 2] = 1.0
    return out


@torch.no_grad()
def robustness_eval(model, dl, noise_fn, name):
    model.eval()
    psnrs = []
    for batch in dl:
        x1 = noise_fn(batch["x1"]).to(device)
        x2 = batch["x2"].to(device)
        out = model(x1)
        x2_hat = out["x2_hat"].clamp(0, 1).cpu().numpy()
        x2_np = x2.cpu().numpy()
        for b in range(x2_np.shape[0]):
            for c in range(x2_np.shape[1]):
                psnrs.append(psnr_metric(x2_np[b, c], x2_hat[b, c], data_range=1.0))
    mean_psnr = float(np.mean(psnrs))
    print(f"{name:45s} PSNR = {mean_psnr:.2f} dB")
    return mean_psnr


records = []
for sigma in [0.0, 0.05, 0.10, 0.20]:
    fn = lambda x, s=sigma: add_gaussian_noise(x, s)
    records.append(("Baseline", sigma, robustness_eval(model_baseline, test_dl, fn, f"Baseline  gaussian sigma={sigma}")))
    records.append(("SCL-World", sigma, robustness_eval(model_scl_latent, test_dl, fn, f"SCL-World gaussian sigma={sigma}")))

rob_df = pd.DataFrame(records, columns=["model", "sigma", "psnr"])
pivot = rob_df.pivot(index="sigma", columns="model", values="psnr")
pivot.plot(marker="o", figsize=(6, 4))
plt.ylabel("PSNR (dB)"); plt.xlabel("Gaussian noise sigma (on study1)")
plt.title("Robustness of study2 prediction to input corruption")
plt.tight_layout(); plt.show()


In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold


def run_kfold_cv(patient_dirs, cfg, k=5, epochs=None, val_frac_within_fold=0.15,
                  demographics_df=None, seeds=None, checkpoint_path="outputs/kfold_checkpoint_2d.csv"):
    '''Runs the Baseline-vs-SCL-Latent comparison independently on each of k
    patient-level folds, and returns both the raw per-fold(-per-seed) results and a
    mean +/- std summary. Uses StratifiedKFold on `cfg["stratify_by"]` (default "ms_type")
    when demographics are available and every class has >= k members -- consistent
    with the primary train/val/test split (Section 3b) -- and falls back to plain
    KFold otherwise (with a printed explanation, same pattern as
    stratified_patient_split; never silently produces a different kind of split
    than what it reports doing).

    `seeds`: list of random seeds to retrain each fold under (default: just
    `[cfg["seed"]]`, i.e. ONE seed per fold -- this is now the default and the
    recommended setting for the headline arXiv number: k folds x 1 seed = k
    trainings per model, with variance coming from fold-to-fold patient
    composition, which is the more meaningful axis of variance for a
    patient-level cross-validation claim anyway. Passing a longer `seeds` list
    (e.g. SEEDS from Section 2b) trains every fold under every seed instead, so
    the summary's std reflects BOTH fold-to-fold and seed-to-seed variance --
    more thorough, but k * len(seeds) trainings per model.

    Performance notes (see the optimization pass on this notebook):
      - Each fold's patients (train+val pool, and test pool) are loaded from disk
        and preprocessed into a SliceCache exactly ONCE, regardless of how many
        seeds are trained on that fold. Only the train/val split *within* the
        pool is reshuffled per seed -- no repeated disk I/O or renormalization.
      - Every (fold, seed) result is appended to `checkpoint_path` as soon as
        it's computed. If this cell is interrupted (timeout, crash, manual
        stop), rerunning it picks up where it left off instead of retraining
        folds/seeds that already finished.
    '''
    seeds = list(seeds) if seeds is not None else [cfg["seed"]]
    nw = cfg.get("num_workers", 0)

    os.makedirs(os.path.dirname(checkpoint_path) or ".", exist_ok=True)
    done_keys = set()
    prev_rows = []
    if os.path.exists(checkpoint_path):
        prev_df = pd.read_csv(checkpoint_path)
        prev_rows = prev_df.to_dict("records")
        done_keys = set(zip(prev_df["fold"], prev_df["seed"], prev_df["model"]))
        print(f"[kfold] resuming from checkpoint: {len(prev_df)} run(s) already completed "
              f"in {checkpoint_path}")

    pids = [os.path.basename(p) for p in patient_dirs]
    pid_to_dir = {os.path.basename(p): p for p in patient_dirs}
    strat_col_name = cfg.get("stratify_by")

    can_stratify = False
    fold_labels = None
    if demographics_df is not None and strat_col_name is not None:
        matches = [c for c in demographics_df.columns if c.lower() == strat_col_name.lower()]
        if matches:
            actual_col = matches[0]
            fold_labels = [demographics_df[actual_col].get(pid) for pid in pids]
            has_missing = any(l is None or (isinstance(l, float) and np.isnan(l)) for l in fold_labels)
            label_counts = pd.Series(fold_labels).value_counts(dropna=False)
            min_class_size = label_counts.min() if len(label_counts) > 0 else 0
            if not has_missing and min_class_size >= k:
                can_stratify = True
            else:
                print(f"[kfold] stratification by '{actual_col}' not viable for k={k} "
                      f"(missing labels: {has_missing}, min class size: {min_class_size} < k={k}) "
                      f"-- falling back to plain KFold.")

    if can_stratify:
        kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=cfg["seed"])
        split_iter = list(kf.split(pids, fold_labels))
        print(f"[kfold] using StratifiedKFold on '{strat_col_name}' (k={k}).")
    else:
        kf = KFold(n_splits=k, shuffle=True, random_state=cfg["seed"])
        split_iter = list(kf.split(pids))
        print(f"[kfold] using plain KFold (k={k}, no stratification).")

    print(f"[kfold] training under {len(seeds)} seed(s) per fold: {seeds}  "
          f"({k} folds x {len(seeds)} seeds = {k * len(seeds)} runs per model)")

    fold_results = list(prev_rows)

    for fold_i, (train_idx, test_idx) in enumerate(split_iter):
        train_val_pids = [pids[i] for i in train_idx]
        test_pids_fold = [pids[i] for i in test_idx]
        train_val_dirs = [pid_to_dir[p] for p in train_val_pids]
        test_dirs = [pid_to_dir[p] for p in test_pids_fold]

        all_done_for_fold = all(
            (fold_i + 1, s, m) in done_keys for s in seeds for m in ("Baseline", "SCL-World")
        )
        if all_done_for_fold:
            print(f"[kfold] fold {fold_i + 1}/{k}: all seeds already in checkpoint -- skipping load.")
            continue

        # Load + precompute this fold's patients ONCE. Reused across every seed below --
        # no re-reading NIfTI files or re-normalizing/re-resizing slices per seed.
        print(f"\n[kfold] fold {fold_i + 1}/{k}: loading {len(train_val_dirs)} train/val "
              f"+ {len(test_dirs)} test patient(s) into cache...")
        train_val_cache = SliceCache(train_val_dirs, cfg, demographics_df=demographics_df)
        test_cache = SliceCache(test_dirs, cfg, demographics_df=demographics_df)

        test_ds_f = MSLongitudinalSliceDataset(test_pids_fold, cfg, train=False, cache=test_cache)
        if len(test_ds_f) == 0:
            print(f"  [skip fold {fold_i + 1}]: empty test split")
            continue
        test_dl_f = DataLoader(test_ds_f, batch_size=cfg["batch_size"], shuffle=False,
                                num_workers=nw, pin_memory=True, persistent_workers=(nw > 0))

        n_val = max(1, int(len(train_val_pids) * val_frac_within_fold))

        for seed_i, s in enumerate(seeds):
            if (fold_i + 1, s, "Baseline") in done_keys and (fold_i + 1, s, "SCL-World") in done_keys:
                print(f"  [kfold] fold {fold_i + 1} seed {s}: already in checkpoint -- skipping")
                continue

            print(f"\n===== Fold {fold_i + 1}/{k}  |  seed {s} ({seed_i + 1}/{len(seeds)}) =====")

            rng_fold = random.Random(s + fold_i)
            shuffled = train_val_pids[:]
            rng_fold.shuffle(shuffled)
            val_pids_fold = shuffled[:n_val]
            train_pids_fold = shuffled[n_val:]

            # Reshuffled *indices* into the already-loaded cache -- no disk access here.
            train_ds_f = MSLongitudinalSliceDataset(train_pids_fold, cfg, train=True, cache=train_val_cache)
            val_ds_f = MSLongitudinalSliceDataset(val_pids_fold, cfg, train=False, cache=train_val_cache)

            if len(train_ds_f) == 0 or len(val_ds_f) == 0:
                print(f"  [skip fold {fold_i + 1} seed {s}]: an empty split "
                      f"(train={len(train_ds_f)}, val={len(val_ds_f)})")
                continue

            train_dl_f = DataLoader(train_ds_f, batch_size=cfg["batch_size"], shuffle=True,
                                     num_workers=nw, pin_memory=True, persistent_workers=(nw > 0),
                                     drop_last=True, generator=torch.Generator().manual_seed(s + fold_i))
            val_dl_f = DataLoader(val_ds_f, batch_size=cfg["batch_size"], shuffle=False,
                                   num_workers=nw, pin_memory=True, persistent_workers=(nw > 0))

            # Re-select (lambda*, alpha*) on THIS fold's training data (avoids leaking
            # test-fold information into the hyperparameter, and matches the Section 4b /
            # 12b policy of never reusing a single global (lambda*, alpha*) everywhere).
            search_f = grid_search_scl_hyperparams(
                train_dl_f, cfg["scl_lambda_grid"], cfg["scl_alpha_grid"], cfg,
                n_batches=cfg["scl_search_batches"],
            )
            best_f = search_f.sort_values("auc", ascending=False).iloc[0]
            scl_cfg_f = dict(lam=float(best_f["lam"]), alpha=float(best_f["alpha"]),
                              dt=cfg["scl_dt"], K=cfg["scl_K_train"], eps=cfg["scl_eps"], xi=cfg["scl_xi"])

            in_ch_f = out_ch_f = len(cfg["modalities"])
            m_base = WorldModelBase(in_ch_f, out_ch_f, latent_ch=cfg["latent_dim"]).to(device)
            m_scl = WorldModelSCLLatent(in_ch_f, out_ch_f, latent_ch=cfg["latent_dim"], scl_cfg=scl_cfg_f).to(device)
            scl_eval_f = SCLFlow(**scl_cfg_f).to(device)

            pw = compute_pos_weight(train_dl_f) if cfg.get("use_pos_weighting", False) else None

            m_base, _ = train_model(m_base, f"fold{fold_i + 1}-seed{s}-Baseline", train_dl_f, val_dl_f, cfg,
                                     epochs=epochs or cfg["epochs"], pos_weight=pw, seed=s)
            m_scl, _ = train_model(m_scl, f"fold{fold_i + 1}-seed{s}-SCL", train_dl_f, val_dl_f, cfg,
                                    epochs=epochs or cfg["epochs"], scl_module_for_loss=scl_eval_f,
                                    pos_weight=pw, seed=s)

            rb = evaluate_model(m_base, test_dl_f, "Baseline")
            rs = evaluate_model(m_scl, test_dl_f, "SCL-World")
            rb["fold"] = fold_i + 1; rb["seed"] = s
            rs["fold"] = fold_i + 1; rs["seed"] = s
            fold_results.append(rb)
            fold_results.append(rs)

            # Checkpoint immediately: append these two rows to disk so a timeout after
            # this point doesn't lose fold/seed combinations already completed.
            pd.DataFrame([rb, rs]).to_csv(
                checkpoint_path, mode="a", header=not os.path.exists(checkpoint_path), index=False
            )
            done_keys.add((fold_i + 1, s, "Baseline"))
            done_keys.add((fold_i + 1, s, "SCL-World"))
            print(f"  [checkpoint] saved fold {fold_i + 1} seed {s} results -> {checkpoint_path}")

    fold_df = pd.DataFrame(fold_results)
    summary = fold_df.groupby("model")[["psnr_mean", "ssim_mean", "dice_mean"]].agg(["mean", "std"])
    return fold_df, summary


if RUN_KFOLD:
    # ARXIV / HEADLINE RUN: full fold x seed factorial (k folds x len(SEEDS) seeds
    # per model). kfold_summary's std now reflects BOTH fold-to-fold AND
    # seed-to-seed variance -- the most defensible number for the paper.
    # This now finishes in a fraction of the original 12+ hour attempt because:
    #   - each fold's patients are loaded + preprocessed into a SliceCache ONCE
    #     and reused across every seed (no repeated disk I/O / renormalization),
    #   - AMP + early stopping cut per-training time substantially, and
    #   - every (fold, seed) result is checkpointed to disk immediately, so an
    #     interruption resumes instead of restarting from scratch.
    # If you just want a quick sanity check instead of the headline number,
    # pass seeds=[CFG["seed"]] for the cheap fold-only-variance version.
    fold_df, kfold_summary = run_kfold_cv(patient_dirs, CFG, k=CFG["kfold_n_splits"],
                                           demographics_df=demographics_df, seeds=SEEDS)
    print("\nPer-fold-per-seed results:")
    print(fold_df[["model", "fold", "seed", "psnr_mean", "ssim_mean", "dice_mean"]])
    print(f"\nAcross-fold x seed summary (patient-level, mean +/- std, "
          f"k={CFG['kfold_n_splits']} folds x {len(SEEDS)} seeds "
          f"= {CFG['kfold_n_splits'] * len(SEEDS)} runs per model):")
    print(kfold_summary)
else:
    fold_df, kfold_summary = None, None
    print("RUN_KFOLD is False -- skipping. Set RUN_KFOLD=True in Section 2b for the "
          "k-fold-validated final number.")

## k-fold x multi-seed sweep, extended: add CurvatureDynamics

Same 25-run (k=5 folds x 5 seeds) protocol as Section 13b's Baseline-vs-SCL-Latent comparison, now including `WorldModelCurvatureDynamics`. This is the number that actually decides whether CurvatureDynamics can be promoted to the paper's primary model: it needs to (a) hold or beat SCL-Latent's Dice under resampling, and (b) not blow up in variance relative to SCL-Latent, since a headline model whose whole point is a stability claim can't be swapped for one with untested stability. Checkpointed separately from Section 13b so a timeout doesn't cost you the already-completed Baseline/SCL-Latent runs.

In [ ]:
def run_kfold_cv_curvature(patient_dirs, cfg, k=5, epochs=None, val_frac_within_fold=0.15,
                            demographics_df=None, seeds=None,
                            checkpoint_path="outputs/kfold_checkpoint_curvature_2d.csv"):
    '''Same design as run_kfold_cv (Section 13b), but trains and evaluates
    WorldModelCurvatureDynamics instead of Baseline/SCL-Latent, under the identical
    fold splits (same random_state) so the two checkpoint files are directly comparable
    row-for-row on (fold, seed). Kept as a separate function/checkpoint rather than
    folded into run_kfold_cv so a re-run of one never touches the other's completed work.'''
    seeds = list(seeds) if seeds is not None else [cfg["seed"]]
    nw = cfg.get("num_workers", 0)

    os.makedirs(os.path.dirname(checkpoint_path) or ".", exist_ok=True)
    done_keys = set()
    prev_rows = []
    if os.path.exists(checkpoint_path):
        prev_df = pd.read_csv(checkpoint_path)
        prev_rows = prev_df.to_dict("records")
        done_keys = set(zip(prev_df["fold"], prev_df["seed"], prev_df["model"]))
        print(f"[kfold-curv] resuming from checkpoint: {len(prev_df)} run(s) already completed "
              f"in {checkpoint_path}")

    pids = [os.path.basename(p) for p in patient_dirs]
    pid_to_dir = {os.path.basename(p): p for p in patient_dirs}
    strat_col_name = cfg.get("stratify_by")

    can_stratify = False
    fold_labels = None
    if demographics_df is not None and strat_col_name is not None:
        matches = [c for c in demographics_df.columns if c.lower() == strat_col_name.lower()]
        if matches:
            actual_col = matches[0]
            fold_labels = [demographics_df[actual_col].get(pid) for pid in pids]
            has_missing = any(l is None or (isinstance(l, float) and np.isnan(l)) for l in fold_labels)
            label_counts = pd.Series(fold_labels).value_counts(dropna=False)
            min_class_size = label_counts.min() if len(label_counts) > 0 else 0
            if not has_missing and min_class_size >= k:
                can_stratify = True

    # Same random_state as run_kfold_cv (cfg["seed"]) => identical fold membership,
    # so Baseline/SCL-Latent and CurvatureDynamics are compared on the same patient splits.
    if can_stratify:
        kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=cfg["seed"])
        split_iter = list(kf.split(pids, fold_labels))
        print(f"[kfold-curv] using StratifiedKFold on '{strat_col_name}' (k={k}).")
    else:
        kf = KFold(n_splits=k, shuffle=True, random_state=cfg["seed"])
        split_iter = list(kf.split(pids))
        print(f"[kfold-curv] using plain KFold (k={k}, no stratification).")

    print(f"[kfold-curv] training under {len(seeds)} seed(s) per fold: {seeds}  "
          f"({k} folds x {len(seeds)} seeds = {k * len(seeds)} runs for CurvatureDynamics)")

    fold_results = list(prev_rows)

    for fold_i, (train_idx, test_idx) in enumerate(split_iter):
        train_val_pids = [pids[i] for i in train_idx]
        test_pids_fold = [pids[i] for i in test_idx]
        train_val_dirs = [pid_to_dir[p] for p in train_val_pids]
        test_dirs = [pid_to_dir[p] for p in test_pids_fold]

        all_done_for_fold = all((fold_i + 1, s, "CurvatureDynamics") in done_keys for s in seeds)
        if all_done_for_fold:
            print(f"[kfold-curv] fold {fold_i + 1}/{k}: all seeds already in checkpoint -- skipping load.")
            continue

        print(f"\n[kfold-curv] fold {fold_i + 1}/{k}: loading {len(train_val_dirs)} train/val "
              f"+ {len(test_dirs)} test patient(s) into cache...")
        train_val_cache = SliceCache(train_val_dirs, cfg, demographics_df=demographics_df)
        test_cache = SliceCache(test_dirs, cfg, demographics_df=demographics_df)

        test_ds_f = MSLongitudinalSliceDataset(test_pids_fold, cfg, train=False, cache=test_cache)
        if len(test_ds_f) == 0:
            print(f"  [skip fold {fold_i + 1}]: empty test split")
            continue
        test_dl_f = DataLoader(test_ds_f, batch_size=cfg["batch_size"], shuffle=False,
                                num_workers=nw, pin_memory=True, persistent_workers=(nw > 0))

        n_val = max(1, int(len(train_val_pids) * val_frac_within_fold))

        for seed_i, s in enumerate(seeds):
            if (fold_i + 1, s, "CurvatureDynamics") in done_keys:
                print(f"  [kfold-curv] fold {fold_i + 1} seed {s}: already in checkpoint -- skipping")
                continue

            print(f"\n===== [CurvatureDynamics] Fold {fold_i + 1}/{k}  |  seed {s} ({seed_i + 1}/{len(seeds)}) =====")

            rng_fold = random.Random(s + fold_i)
            shuffled = train_val_pids[:]
            rng_fold.shuffle(shuffled)
            val_pids_fold = shuffled[:n_val]
            train_pids_fold = shuffled[n_val:]

            train_ds_f = MSLongitudinalSliceDataset(train_pids_fold, cfg, train=True, cache=train_val_cache)
            val_ds_f = MSLongitudinalSliceDataset(val_pids_fold, cfg, train=False, cache=train_val_cache)

            if len(train_ds_f) == 0 or len(val_ds_f) == 0:
                print(f"  [skip fold {fold_i + 1} seed {s}]: an empty split "
                      f"(train={len(train_ds_f)}, val={len(val_ds_f)})")
                continue

            train_dl_f = DataLoader(train_ds_f, batch_size=cfg["batch_size"], shuffle=True,
                                     num_workers=nw, pin_memory=True, persistent_workers=(nw > 0),
                                     drop_last=True, generator=torch.Generator().manual_seed(s + fold_i))
            val_dl_f = DataLoader(val_ds_f, batch_size=cfg["batch_size"], shuffle=False,
                                   num_workers=nw, pin_memory=True, persistent_workers=(nw > 0))

            # Re-select (lambda*, alpha*) on THIS fold's training data -- same policy as
            # run_kfold_cv, so CurvatureDynamics gets the same treatment, not a shortcut.
            search_f = grid_search_scl_hyperparams(
                train_dl_f, cfg["scl_lambda_grid"], cfg["scl_alpha_grid"], cfg,
                n_batches=cfg["scl_search_batches"],
            )
            best_f = search_f.sort_values("auc", ascending=False).iloc[0]
            scl_cfg_f = dict(lam=float(best_f["lam"]), alpha=float(best_f["alpha"]),
                              dt=cfg["scl_dt"], K=cfg["scl_K_train"], eps=cfg["scl_eps"], xi=cfg["scl_xi"])

            in_ch_f = out_ch_f = len(cfg["modalities"])
            m_curv = WorldModelCurvatureDynamics(in_ch_f, out_ch_f, latent_ch=cfg["latent_dim"],
                                                  scl_cfg=scl_cfg_f).to(device)
            scl_eval_f = SCLFlow(**scl_cfg_f).to(device)
            pw = compute_pos_weight(train_dl_f) if cfg.get("use_pos_weighting", False) else None

            m_curv, _ = train_model(m_curv, f"fold{fold_i + 1}-seed{s}-CurvatureDynamics",
                                     train_dl_f, val_dl_f, cfg, epochs=epochs or cfg["epochs"],
                                     scl_module_for_loss=scl_eval_f, pos_weight=pw, seed=s)

            rc = evaluate_model(m_curv, test_dl_f, "CurvatureDynamics")
            rc["fold"] = fold_i + 1; rc["seed"] = s
            fold_results.append(rc)

            pd.DataFrame([rc]).to_csv(
                checkpoint_path, mode="a", header=not os.path.exists(checkpoint_path), index=False
            )
            done_keys.add((fold_i + 1, s, "CurvatureDynamics"))
            print(f"  [checkpoint] saved fold {fold_i + 1} seed {s} CurvatureDynamics result -> {checkpoint_path}")

    fold_df_curv = pd.DataFrame(fold_results)
    summary_curv = fold_df_curv.groupby("model")[["psnr_mean", "ssim_mean", "dice_mean"]].agg(["mean", "std"])
    return fold_df_curv, summary_curv


if RUN_KFOLD:
    curvature_fold_df, curvature_kfold_summary = run_kfold_cv_curvature(
        patient_dirs, CFG, k=CFG["kfold_n_splits"], demographics_df=demographics_df, seeds=SEEDS
    )
    print("\nPer-fold-per-seed results [CurvatureDynamics]:")
    print(curvature_fold_df[["model", "fold", "seed", "psnr_mean", "ssim_mean", "dice_mean"]])
    print(f"\nAcross-fold x seed summary [CurvatureDynamics] "
          f"(k={CFG['kfold_n_splits']} folds x {len(SEEDS)} seeds = "
          f"{CFG['kfold_n_splits'] * len(SEEDS)} runs):")
    print(curvature_kfold_summary)

    # Combined k-fold summary: Baseline vs SCL-Latent vs CurvatureDynamics, same protocol.
    combined_kfold_summary = pd.concat(
        [kfold_summary, curvature_kfold_summary]
    ) if kfold_summary is not None else curvature_kfold_summary
    print("\n=== Combined k-fold x multi-seed summary: Baseline vs SCL-Latent vs CurvatureDynamics ===")
    print(combined_kfold_summary)

    # Decision check for the paper's headline-model question -- printed, not auto-decided:
    # promoting CurvatureDynamics requires it to both match/beat SCL-Latent's validated Dice
    # AND not have materially worse cross-run variance, under this SAME protocol.
    try:
        dice_scl_mean = kfold_summary.loc["SCL-World", ("dice_mean", "mean")]
        dice_scl_std = kfold_summary.loc["SCL-World", ("dice_mean", "std")]
        dice_curv_mean = curvature_kfold_summary.loc["CurvatureDynamics", ("dice_mean", "mean")]
        dice_curv_std = curvature_kfold_summary.loc["CurvatureDynamics", ("dice_mean", "std")]
        print(f"\n[decision check] SCL-Latent Dice: {dice_scl_mean:.3f} +/- {dice_scl_std:.3f}  "
              f"(k-fold x multi-seed, n=25 runs)")
        print(f"[decision check] CurvatureDynamics Dice: {dice_curv_mean:.3f} +/- {dice_curv_std:.3f}  "
              f"(k-fold x multi-seed, n=25 runs)")
        if dice_curv_mean >= dice_scl_mean and dice_curv_std <= dice_scl_std * 1.25:
            print("[decision check] CurvatureDynamics matches/beats SCL-Latent on Dice with "
                  "comparable or better stability under this protocol -- promoting it to the "
                  "primary model would now be evidence-backed, not a relabel.")
        else:
            print("[decision check] CurvatureDynamics does NOT clearly beat SCL-Latent under this "
                  "protocol (either lower mean Dice, or meaningfully higher variance, or both). "
                  "Report both variants transparently -- this is itself a legitimate and useful "
                  "finding for the paper's ablation section, consistent with how Section 6.2 "
                  "already treats protocol-dependent reversals.")
    except KeyError as e:
        print(f"[decision check] could not compute (missing key {e}) -- inspect the summaries above manually.")
else:
    curvature_fold_df, curvature_kfold_summary, combined_kfold_summary = None, None, None
    print("RUN_KFOLD is False -- skipping CurvatureDynamics k-fold sweep too.")


In [ ]:
os.makedirs("outputs", exist_ok=True)

torch.save(model_baseline.state_dict(), "outputs/world_model_baseline_2d.pt")
torch.save(model_scl_latent.state_dict(), "outputs/world_model_scl_latent_2d.pt")

with open("outputs/results_2d.json", "w") as f:
    json.dump({"baseline": res_baseline, "scl_world": res_scl,
                "baseline_slice_weighted": res_baseline_slice, "scl_world_slice_weighted": res_scl_slice,
                "scl_hyperparams_selected": scl_cfg}, f, indent=2)

with open("outputs/config_2d.json", "w") as f:
    json.dump(CFG, f, indent=2)

# Full per-epoch training histories -- saved separately from results so you can
# regenerate publication-quality loss/Dice curves later without re-training.
with open("outputs/history_baseline_2d.json", "w") as f:
    json.dump(hist_baseline, f, indent=2)
with open("outputs/history_scl_2d.json", "w") as f:
    json.dump(hist_scl, f, indent=2)

if "stats_df" in dir():
    stats_df.to_csv("outputs/stats_2d.csv")
if "topo_baseline" in dir():
    topo_df.to_csv("outputs/topology_2d.csv")

# Multi-seed robustness study (Section 12b) -- per-seed rows + mean/std summary.
if multiseed_df is not None:
    multiseed_df.to_csv("outputs/multiseed_results_2d.csv", index=False)
    multiseed_summary.to_csv("outputs/multiseed_summary_2d.csv")
    print("Saved outputs/multiseed_results_2d.csv and outputs/multiseed_summary_2d.csv")

# K-fold cross-validation (Section 13b) -- the headline result -- per-fold rows +
# mean/std summary.
if fold_df is not None:
    fold_df.to_csv("outputs/kfold_results_2d.csv", index=False)
    kfold_summary.to_csv("outputs/kfold_summary_2d.csv")
    print("Saved outputs/kfold_results_2d.csv and outputs/kfold_summary_2d.csv")

print("Saved model weights, config, results, and full training histories to ./outputs/")

# CurvatureDynamics validation sweep (multi-seed + k-fold x multi-seed) --
# saved separately so the decision of whether it becomes the primary model
# is backed by the same kind of evidence as SCL-Latent's stability claim.
if curvature_multiseed_df is not None:
    curvature_multiseed_df.to_csv("outputs/curvature_multiseed_results_2d.csv", index=False)
    curvature_multiseed_summary.to_csv("outputs/curvature_multiseed_summary_2d.csv")
    combined_multiseed_summary.to_csv("outputs/combined_multiseed_summary_2d.csv")
    print("Saved outputs/curvature_multiseed_results_2d.csv, curvature_multiseed_summary_2d.csv, combined_multiseed_summary_2d.csv")

if curvature_fold_df is not None:
    curvature_fold_df.to_csv("outputs/curvature_kfold_results_2d.csv", index=False)
    curvature_kfold_summary.to_csv("outputs/curvature_kfold_summary_2d.csv")
    combined_kfold_summary.to_csv("outputs/combined_kfold_summary_2d.csv")
    print("Saved outputs/curvature_kfold_results_2d.csv, curvature_kfold_summary_2d.csv, combined_kfold_summary_2d.csv")


In [ ]:
def plot_full_history(hist_a, hist_b, name_a="Baseline", name_b="SCL-World", save_path=None):
    '''Publication-style figure: train+val loss and val Dice, side by side.
    Pass save_path (e.g. "outputs/training_curves_2d.png") to also save a copy;
    if None, only displays inline.'''
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    for split, ls in [("train", "--"), ("val", "-")]:
        axes[0].plot([h["total"] for h in hist_a[split]], ls, color="#888888",
                     label=f"{name_a} ({split})")
        axes[0].plot([h["total"] for h in hist_b[split]], ls, color="#0053a5",
                     label=f"{name_b} ({split})")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("total loss"); axes[0].legend(fontsize=8)
    axes[0].set_title("Training and validation loss")

    axes[1].plot([1 - h["lesion_dice"] for h in hist_a["val"]], color="#888888", label=name_a)
    axes[1].plot([1 - h["lesion_dice"] for h in hist_b["val"]], color="#0053a5", label=name_b)
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation Dice"); axes[1].legend(fontsize=8)
    axes[1].set_title("Validation Dice over training")

    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


plot_full_history(hist_baseline, hist_scl, save_path="outputs/training_curves_2d.png")
print("Saved outputs/training_curves_2d.png")
